# Image Editing LLM Pipeline -- Phase 1: VLM QLoRA Fine-Tuning
**Spec version:** 1.7 | **Notebook version:** 1.5 | **Target runtime:** Google Colab A100
**Phase in this notebook:** Phase 1 -- Qwen2.5-VL-3B QLoRA fine-tuning for structured JSON edit prediction
**Depends on:** Phase 0 notebook (v2.10)

**v1.3 changes vs v1.2:**
- `route_bbox` removed; replaced by `select_target_bbox` (class-name matching, clip_score tiebreaker, area fallback)
- `build_messages` now includes ALL segmentation regions in the user turn so the model sees full scene context
- Ground-truth bbox selection uses class-name ↔ instruction matching, not area heuristic
- `annotations` and `seg_dims` stored in every sample dict and propagated to tokeniser and hidden-state extractor
- `build_messages` is the single canonical prompt builder used by training, hidden-state cache, and inference


## S0 -- Global Configuration
*Identical to Phase 0 S0.1 -- copy any CFG changes here.*


In [ ]:
# -- S0.1  GLOBAL CONFIG -- identical to Phase 0; edit this cell only -----------
# All downstream cells import from CFG. Never hardcode paths elsewhere.

import os, sys, json
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

@dataclass
class PipelineConfig:
    # -- Drive root ----------------------------------------------------------
    drive_root: Path = Path("/content/drive/MyDrive/img_edit_pipeline")

    # -- Dataset -------------------------------------------------------------
    hf_dataset_id: str         = "sysuyy/ImgEdit"
    hf_configs: Optional[list] = None
    n_subset: int              = 10_000

    # Confirmed singleturn edit types (Phase 0 S2.1 -- expanded beyond spec)
    expected_edit_types: tuple = (
        "action", "add", "adjust", "background", "content",
        "hybrid", "reference", "remove", "replace", "style", "version",
    )

    mask_dataset_id: str = "sysuyy/ImgEdit_recap_mask"

    # Bbox routing keywords (global-adjust classification)
    global_adjust_keywords: tuple = (
        "lighting", "light", "brightness", "exposure", "contrast",
        "saturation", "hue", "tone", "tones", "overall", "scene",
        "atmosphere", "color temperature", "white balance",
    )

    # -- Phase 1 model -------------------------------------------------------
    vlm_model_id: str       = "Qwen/Qwen2.5-VL-3B-Instruct"
    vlm_quant_bits: int     = 4
    vlm_lora_r: int         = 16
    vlm_lora_alpha: int     = 32
    vlm_lora_dropout: float = 0.05
    # Hidden dim verified at load time -- updated if model differs
    vlm_hidden_dim: int     = 2048   # Qwen2.5-VL-3B confirmed hidden size

    # -- Phase 2 model -------------------------------------------------------
    sd_model_id: str       = "runwayml/stable-diffusion-inpainting"
    sd_cross_attn_dim: int = 768   # SD 1.5 cross-attention dim (constant)
    sd_lora_r: int         = 8
    sd_lora_alpha: int     = 16

    # -- Training ------------------------------------------------------------
    phase1_epochs: int         = 3
    phase1_batch_size: int     = 2
    phase1_grad_accum: int     = 8
    phase1_lr: float           = 2e-4
    phase1_max_seq_len: int    = 2560 # Increased from 1024
    phase1_max_img_pixels: int = 802_816   # 28*28*1024
    phase1_warmup_steps: int   = 100
    phase1_eval_steps: int     = 200

    phase2_epochs: int         = 5
    phase2_batch_size: int     = 1
    phase2_grad_accum: int     = 16
    phase2_lr: float           = 1e-4
    phase2_warmup_steps: int   = 200
    phase2_shard_size: int     = 5_000   # samples per hidden-state shard

    # -- Audit thresholds ----------------------------------------------------
    max_invalid_rle_frac: float = 0.05
    max_fallback_frac: float    = 0.15

    # -- Derived paths -------------------------------------------------------
    @property
    def data_dir(self) -> Path:
        return self.drive_root / "data" / "imgedit_subset"

    @property
    def benchmark_dir(self) -> Path:
        return self.drive_root / "data" / "benchmark"

    @property
    def ckpt_phase1(self) -> Path:
        return self.drive_root / "checkpoints" / "phase1_vlm"

    @property
    def ckpt_phase2(self) -> Path:
        return self.drive_root / "checkpoints" / "phase2_diffusion"

    @property
    def outputs_eval(self) -> Path:
        return self.drive_root / "outputs" / "eval"

    @property
    def outputs_scores(self) -> Path:
        return self.drive_root / "outputs" / "scores"

    @property
    def manifest_path(self) -> Path:
        return self.data_dir / "samples.json"

    @property
    def filtered_manifest_path(self) -> Path:
        # Phase 0 S7.3 output -- used for all Phase 1 and Phase 2 training.
        # Phase 0 already filtered out: mask_loaded=False and full-image bbox samples.
        return self.data_dir / "samples_filtered.json"

    @property
    def hidden_states_dir(self) -> Path:
        # Phase 1 output -- VLM hidden-state shards for Phase 2 conditioning
        return self.data_dir / "vlm_hidden_states"

    @property
    def hidden_states_dir_v2(self) -> Path:
        # v1.5: separate Drive directory for hidden states extracted *after*
        # resumed Phase 1 fine-tuning.  Lets us keep the original v1 cache
        # intact for A/B comparison and prevents the cache_vlm_hidden_states
        # restart-skip logic from mixing shards produced by different adapters.
        return self.data_dir / "vlm_hidden_states_v2"

    @property
    def filtered_manifest_v2_path(self) -> Path:
        # v1.5: dedicated manifest for the v2 cache.  Built by S11.4a as a
        # deep copy of samples_filtered.json with shard_id/row_index reset
        # to None on every entry so cache_vlm_hidden_states does not skip.
        return self.data_dir / "samples_filtered_v2.json"

    @property
    def audit_path(self) -> Path:
        return self.data_dir / "audit_report.json"

    @property
    def phase1_adapter_dir(self) -> Path:
        # Default save path for the Phase 1 LoRA adapter (used by S10.2 and
        # all downstream cells that reload the adapter).  Sits under the
        # phase1_vlm checkpoint dir so it's easy to find next to any other
        # checkpoint artifacts.
        #
        # IMPORTANT: if your *prior* session saved the adapter to a different
        # path (e.g. you set this manually), either move the files into this
        # directory on Drive, or override AFTER CFG is instantiated, e.g.:
        #
        #     from pathlib import Path
        #     object.__setattr__(CFG, "_phase1_adapter_dir_override",
        #                        Path("/content/drive/MyDrive/.../my_adapter"))
        #
        # ...or simpler: just monkey-patch the property by reassigning the
        # attribute on the dataclass instance (works because we use @property,
        # not __slots__).  The S11.4 discovery cell below scans Drive for
        # adapter_config.json files and tells you what to override to.
        return self.ckpt_phase1 / "lora_adapter"


CFG = PipelineConfig()
print("Config loaded.")
print(f"  Drive root         : {CFG.drive_root}")
print(f"  Data dir           : {CFG.data_dir}")
print(f"  Filtered manifest  : {CFG.filtered_manifest_path}")
print(f"  Hidden states dir  : {CFG.hidden_states_dir}")
print(f"  Hidden states v2   : {CFG.hidden_states_dir_v2}")
print(f"  Filtered manifest v2: {CFG.filtered_manifest_v2_path}")
print(f"  Phase 1 checkpoint : {CFG.ckpt_phase1}")
print(f"  VLM model          : {CFG.vlm_model_id}")
print(f"  n_subset           : {CFG.n_subset:,}")


In [ ]:
# -- S0.2  Mount Drive and verify directory structure -------------------------
# Re-runnable: mounting an already-mounted Drive is a no-op.

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def scaffold_dirs(cfg) -> None:
    dirs = [
        cfg.data_dir,
        cfg.benchmark_dir,
        cfg.ckpt_phase1,
        cfg.ckpt_phase2,
        cfg.outputs_eval,
        cfg.outputs_scores,
        cfg.hidden_states_dir,
    ]
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)
    print(f"Drive scaffold verified ({len(dirs)} directories).")
    for d in dirs:
        print(f"  ok  {d}")

scaffold_dirs(CFG)

# Verify Phase 0 output exists before doing any Phase 1 work
if not CFG.filtered_manifest_path.exists():
    raise FileNotFoundError(
        f"samples_filtered.json not found at {CFG.filtered_manifest_path}\n"
        "Run Phase 0 notebook (v2.10) through S7.3 before starting Phase 1."
    )

import json as _json_check
_manifest_check = _json_check.loads(CFG.filtered_manifest_path.read_text())
print(f"\nok  samples_filtered.json: {len(_manifest_check):,} samples")
del _manifest_check, _json_check


## S1.1 -- Package Installation (run once per runtime)
Warning: If Phase 0 S1.1 was already run in this runtime, skip this cell.
Run it only if starting a fresh Colab session.


In [ ]:
# -- S1.1  Install pinned dependencies (same versions as Phase 0) -------------
# Installs Phase 1 extras not in Phase 0's S1.1 (primarily qwen-vl-utils).
# Re-running is safe -- pip silently skips already-installed packages.

import subprocess, sys

def pip_install(packages, extra_flags=""):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + packages
    if extra_flags:
        cmd += extra_flags.split()
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"pip install failed: {packages}")
    print(f"  installed: {packages}")

# Phase 0 core packages (skip if already installed)
print("Checking core packages...")
pip_install([
    "transformers==4.49.0",
    "peft==0.12.0",
    "trl==0.11.0",
    "bitsandbytes==0.45.4",
    "accelerate==0.34.0",
    "diffusers==0.30.0",
    "datasets==2.21.0",
    "safetensors",
    "pycocotools",
    "tqdm",
])

# Phase 1 extra: qwen-vl-utils provides process_vision_info for image handling
print("\nInstalling Phase 1 extras...")
pip_install(["qwen-vl-utils"])

print("\nAll packages installed. Restart kernel if prompted, then run S1.2.")


## S1.2 -- Import and Version Sanity Checks
*Run after every kernel restart. Must pass before any Phase 1 cell runs.*


In [ ]:
# -- S1.2  Phase 1 import and version gate ------------------------------------
# Raises on any version mismatch. Must pass before any Phase 1 cell runs.

import importlib, importlib.metadata, sys
from packaging.version import Version

REQUIRED = {
    "torch":          ("2.5.1",  None),
    "transformers":   ("4.49.0", None),
    "peft":           ("0.12.0", "0.13.0"),
    "trl":            ("0.11.0", "0.12.0"),
    "bitsandbytes":   ("0.45.4", None),
    "accelerate":     ("0.34.0", None),
    "diffusers":      ("0.30.0", "0.31.0"),
    "datasets":       ("2.21.0", None),
    "pycocotools":    ("2.0.0",  None),
}

failures = []
for pkg, (vmin, vmax) in REQUIRED.items():
    try:
        ver = Version(importlib.metadata.version(pkg))
        if ver < Version(vmin):
            failures.append(f"  FAIL  {pkg} {ver} < required {vmin}")
        elif vmax and ver >= Version(vmax):
            failures.append(f"  FAIL  {pkg} {ver} >= ceiling {vmax}")
        else:
            print(f"  ok  {pkg} {ver}")
    except Exception as e:
        failures.append(f"  FAIL  {pkg}: {e}")

if failures:
    raise EnvironmentError("Version check FAILED:\n" + "\n".join(failures))

# -- CUDA ------------------------------------------------------------------
import torch
if not torch.cuda.is_available():
    raise EnvironmentError("CUDA not available -- switch runtime to GPU (A100).")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"  ok  CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)} | {vram_gb:.1f} GB VRAM")
if vram_gb < 24:
    print(f"  WARN  VRAM {vram_gb:.1f} GB < 24 GB -- Phase 1 training may OOM. Reduce batch_size.")

# -- Qwen2.5-VL class ------------------------------------------------------
try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    print("  ok  Qwen2_5_VLForConditionalGeneration importable")
except ImportError as e:
    raise EnvironmentError(f"Qwen2.5-VL class not found -- need transformers>=4.49.0: {e}")

# -- PEFT / TRL ------------------------------------------------------------
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments, BitsAndBytesConfig
print("  ok  peft, trl, transformers training classes importable")

# -- qwen-vl-utils ---------------------------------------------------------
try:
    from qwen_vl_utils import process_vision_info as _qvl_pvf
    _HAVE_QWEN_VL_UTILS = True
    print("  ok  qwen_vl_utils.process_vision_info available")
except ImportError:
    _HAVE_QWEN_VL_UTILS = False
    print("  WARN  qwen_vl_utils not found -- using built-in fallback (run S1.1)")

print("\nS1.2 import gate passed. Ready for Phase 1.")


## S3.1 -- Core Utilities (Phase 1 subset)
Subset of Phase 0 S3.1 needed for Phase 1. No HF download code included here.


In [ ]:
# -- S3.1  Core utilities -- Phase 1 subset -----------------------------------
# Included from Phase 0 S3.1:
#   decode_rle_mask, select_target_bbox, bbox helpers, load_json, save_json
# NOT included here (Phase 0 only):
#   HF download helpers, split-tar streaming, build_mask_index
#
# v1.3 change: route_bbox REMOVED.
#   Replaced by select_target_bbox, which uses class-name matching against the
#   edit instruction as the primary selection key.  clip_score is a tiebreaker
#   only when multiple annotations share the same class_name.  Area-based
#   selection is the last resort fallback (with a logged warning).
#   Rationale: route_bbox pre-selected a single bbox via area heuristic,
#   removing the spatial reasoning task the VLM is meant to learn.
#   select_target_bbox keeps the full scene in the model's context and lets
#   the supervision signal come from class-name alignment.

import io, json, logging
import numpy as np
from pathlib import Path
from typing import Any, Optional
from PIL import Image

# -- Logging ---------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger("pipeline")

# -- RLE decode ------------------------------------------------------------
import pycocotools.mask as mask_utils

def decode_rle_mask(rle: dict) -> Optional[np.ndarray]:
    '''Decode a COCO-format RLE dict to a binary uint8 mask (H x W).

    Handles both compressed RLE (counts as str) and uncompressed (counts as list).
    Returns None on any failure -- callers treat None as invalid_rle.
    '''
    if rle is None:
        return None
    try:
        rle_copy = dict(rle)
        if isinstance(rle_copy.get("counts"), str):
            rle_copy["counts"] = rle_copy["counts"].encode("utf-8")
        mask = mask_utils.decode(rle_copy)
        assert mask.ndim == 2, f"Expected 2-D mask, got {mask.shape}"
        return mask
    except Exception as e:
        log.debug(f"RLE decode failed: {e}")
        return None


# -- Edit-type set ---------------------------------------------------------
KNOWN_EDIT_TYPES = set(CFG.expected_edit_types)

# -- Bbox utilities --------------------------------------------------------
def bbox_area_absolute(bbox_xyxy: list) -> float:
    x1, y1, x2, y2 = bbox_xyxy
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def bbox_is_out_of_bounds(bbox_xyxy: list, W: int, H: int) -> bool:
    '''Return True if any coordinate of bbox_xyxy falls outside [0,W] x [0,H].'''
    x1, y1, x2, y2 = bbox_xyxy
    return (x1 < 0 or y1 < 0 or x2 > W or y2 > H)

def clip_bbox_to_image(bbox_xyxy: list, W: int, H: int) -> list:
    '''Clip absolute xyxy bbox to the image boundary [0,W] x [0,H].'''
    x1, y1, x2, y2 = bbox_xyxy
    x1 = max(0, min(x1, W))
    y1 = max(0, min(y1, H))
    x2 = max(0, min(x2, W))
    y2 = max(0, min(y2, H))
    return [x1, y1, x2, y2]

def bbox_to_relative(bbox_xyxy: list, W: int, H: int) -> list:
    '''Convert absolute xyxy bbox to Qwen [0,1000] relative scale.

    Clips to image bounds before normalizing so the output is guaranteed
    to be in [0,1000].
    '''
    x1, y1, x2, y2 = clip_bbox_to_image(bbox_xyxy, W, H)
    return [round(x1/W*1000), round(y1/H*1000), round(x2/W*1000), round(y2/H*1000)]

def bbox_relative_to_absolute(bbox_rel: list, W: int, H: int) -> list:
    '''Convert [0,1000] relative bbox back to absolute pixel coords.'''
    x1, y1, x2, y2 = bbox_rel
    return [round(x1/1000*W), round(y1/1000*H), round(x2/1000*W), round(y2/1000*H)]


# -- Target bbox selection (v1.3 -- class-name matching) -------------------
# Replaces route_bbox (v1.2 area heuristic).
#
# Selection priority:
#   1. Annotations whose class_name appears as a substring of the instruction.
#      (case-insensitive; "shirt" matches "change the shirt to blue")
#   2. If multiple annotations share the same class_name match, break ties with
#      clip_score (higher = more semantically aligned).  If clip_score is absent,
#      break ties by largest bbox area.
#   3. If NO annotation class_name appears in the instruction (unlikely after
#      Phase 0 filtering, but handled defensively), fall back to the largest
#      object-region annotation and log a WARNING.
#
# Why NOT re-use the area heuristic as primary:
#   Area selects the largest region regardless of what the instruction refers to.
#   For "change the shirt to blue" in a scene with a large "person" annotation
#   and a smaller "shirt" annotation, area would always return "person".
#   Class-name matching returns "shirt" -- which is correct supervision.
#
# Coordinate space: img_W / img_H must be the seg-file coordinate space
#   (from seg_data["resolution"]).  Callers are responsible for this.

GLOBAL_ADJUST_KEYWORDS = set(CFG.global_adjust_keywords)

def select_target_bbox(
    edit_type: str,
    edit_description: str,
    annotations: list,
    img_W: int,
    img_H: int,
) -> tuple:
    '''Select the ground-truth target bbox for one training sample.

    Args:
        edit_type        : e.g. "adjust", "add", "remove", ...
        edit_description : natural-language instruction
        annotations      : list of annotation dicts from the seg JSON.
        img_W, img_H     : dimensions of the SEG FILE coordinate space
                           (from seg_data["resolution"]).

    Returns:
        (bbox_rel, matched_by)  where:
            bbox_rel   -- [x1,y1,x2,y2] in [0,1000] relative scale
            matched_by -- "global" | "class_name" | "clip_score" | "area_fallback"
                          describes which selection path was taken (for logging)

    Sentinel [0,0,1000,1000] with matched_by="global" = whole-image operation.
    '''
    FULL_IMAGE = [0, 0, 1000, 1000]
    et = (edit_type or "").strip().lower()

    # -- Global edits: no localised region -----------------------------------
    if et in {"style", "background"}:
        return FULL_IMAGE, "global"
    if et == "adjust":
        desc_lower = (edit_description or "").lower()
        if any(kw in desc_lower for kw in GLOBAL_ADJUST_KEYWORDS):
            return FULL_IMAGE, "global"

    instruction_lower = (edit_description or "").lower()
    all_anns = list(annotations or [])

    if not all_anns:
        return FULL_IMAGE, "area_fallback"

    # -- Step 1: class-name matching -----------------------------------------
    # Find all annotations whose class_name is a non-empty substring of the
    # instruction.  We match substrings (not full words) to handle plurals and
    # compound names; Phase 0 annotation class names are single nouns/phrases.
    matched_anns = []
    for ann in all_anns:
        class_name = (ann.get("class_name") or "").strip().lower()
        if class_name and class_name in instruction_lower:
            matched_anns.append(ann)

    if matched_anns:
        if len(matched_anns) == 1:
            best_ann   = matched_anns[0]
            matched_by = "class_name"
        else:
            # -- Step 2: tiebreaker among class-name matches -----------------
            # Primary tiebreaker: clip_score (higher = better).
            # Secondary tiebreaker: area (larger = safer choice if scores tied).
            # We use clip_score over area because clip_score reflects semantic
            # alignment between the annotation and the edit instruction,
            # whereas area would pick the wrong region when a large but
            # semantically irrelevant object shares the same class name.
            scored = []
            for ann in matched_anns:
                clip_score = ann.get("clip_score")  # float or None
                raw_bbox   = ann.get("bbox") or []
                area = bbox_area_absolute(raw_bbox) if len(raw_bbox) == 4 else 0.0
                # Sort key: (has_score, score_value, area) -- all descending
                scored.append((clip_score is not None, clip_score or 0.0, area, ann))
            scored.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
            best_ann   = scored[0][3]
            matched_by = "clip_score" if scored[0][0] else "class_name"

        raw_bbox = best_ann.get("bbox") or []
        if len(raw_bbox) == 4 and bbox_area_absolute(raw_bbox) > 0:
            return bbox_to_relative(raw_bbox, img_W, img_H), matched_by

    # -- Step 3: area fallback (no class-name matched) -----------------------
    # This path should be rare after Phase 0 filtering.  We log a WARNING so
    # any systematic mismatch between class names and instructions is visible.
    log.debug(
        f"select_target_bbox: no class-name match in instruction "
        f"{instruction_lower[:60]!r} -- falling back to area selection."
    )
    object_anns    = [a for a in all_anns if a.get("region") == "object"]
    candidate_anns = object_anns if object_anns else all_anns

    best_bbox, best_area = None, 0.0
    for ann in candidate_anns:
        raw_bbox = ann.get("bbox")
        if raw_bbox is None or len(raw_bbox) != 4:
            continue
        area = bbox_area_absolute(raw_bbox)
        if area > best_area:
            best_area = area
            best_bbox = raw_bbox

    if best_bbox is None or best_area == 0.0:
        return FULL_IMAGE, "area_fallback"

    return bbox_to_relative(best_bbox, img_W, img_H), "area_fallback"


# -- File I/O helpers ------------------------------------------------------
def save_json(obj: Any, path: Path, indent: int = 2) -> None:
    '''Atomic JSON write (write to tmp then rename).'''
    path = Path(path)
    tmp  = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=indent, ensure_ascii=False))
    tmp.rename(path)

def load_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Expected JSON not found: {path}")
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupt JSON at {path}: {e}") from e


print("S3.1 core utilities loaded (v1.3 -- class-name bbox selection):")
print("  decode_rle_mask, select_target_bbox (replaces route_bbox)")
print("  bbox_to_relative (clips to image bounds), bbox_relative_to_absolute")
print("  clip_bbox_to_image, bbox_is_out_of_bounds")
print("  load_json, save_json")
print(f"  KNOWN_EDIT_TYPES: {sorted(KNOWN_EDIT_TYPES)}")

# -- Smoke test select_target_bbox -----------------------------------------
_test_anns = [
    {"class_name": "shirt",  "bbox": [300, 400, 700, 900], "region": "object", "clip_score": 0.82},
    {"class_name": "person", "bbox": [100, 50,  900, 1200], "region": "object", "clip_score": 0.61},
    {"class_name": "background", "bbox": [0, 0, 1920, 1080], "region": "background"},
]
_bbox, _by = select_target_bbox("adjust", "change the shirt to blue", _test_anns, 1920, 1080)
assert _by == "class_name", f"Expected class_name match, got {_by}"
assert _bbox == bbox_to_relative([300, 400, 700, 900], 1920, 1080), f"Wrong bbox: {_bbox}"

# Tiebreaker test: two annotations with same class_name, different clip_score
_tie_anns = [
    {"class_name": "cat", "bbox": [10, 10, 200, 200], "clip_score": 0.90},
    {"class_name": "cat", "bbox": [300, 300, 800, 800], "clip_score": 0.75},
]
_bbox2, _by2 = select_target_bbox("remove", "remove the cat", _tie_anns, 1000, 1000)
assert _by2 == "clip_score", f"Expected clip_score tiebreak, got {_by2}"
assert _bbox2 == bbox_to_relative([10, 10, 200, 200], 1000, 1000), f"Wrong tiebreak bbox: {_bbox2}"

# Fallback test: no class_name match -> area fallback
_fb_anns = [
    {"class_name": "tree",  "bbox": [0, 0, 500, 1000], "region": "object"},
    {"class_name": "sky",   "bbox": [0, 0, 1920, 400],  "region": "object"},
]
_bbox3, _by3 = select_target_bbox("adjust", "make the clouds fluffier", _fb_anns, 1920, 1000)
assert _by3 == "area_fallback", f"Expected area_fallback, got {_by3}"
print("  ok  select_target_bbox: class_name, clip_score tiebreak, area fallback")

## S8 -- Phase 1: Dataset Formatting

`build_vlm_dataset()` reads `samples_filtered.json`, selects target bboxes via
`select_target_bbox()` (class-name matching), and returns a list of raw training
metadata dicts that include the full annotation list for each sample.

Images are loaded lazily in `Phase1Dataset.__getitem__()` to avoid GPU OOM.

**v1.3:** All segmentation regions are included in the VLM user-turn prompt so
the model must learn to identify the correct region from the scene context.
This is the core spatial reasoning capability being trained.


In [ ]:
# -- S8.1  Phase 1 prompt template constants -- SINGLE definition (v1.3) ------
#
# CRITICAL: These constants are the sole definition of the Phase 1 prompt.
# They MUST be used identically in:
#   - Phase 1 training data tokenisation (S9.4 tokenize_sample)
#   - Phase 1 hidden-state extraction (S11.1 extract_hidden_state_for_sample)
#   - Phase 3 inference (import or copy verbatim -- do NOT reimplement)
# Any deviation causes a training/inference distribution mismatch.
#
# v1.3 change: build_messages now includes ALL segmentation regions in the user
# turn so the model sees the full scene.  The model must learn to associate the
# edit instruction with the correct region -- that association is the capability
# being trained.  Hiding the scene context (v1.2 route_bbox approach) removed
# the reasoning task and produced weaker supervision.
#
# Region bbox format in the prompt: [0,1000] relative coordinates.
# Why [0,1000] (not pixel): the target output is also [0,1000], so the model
# can learn to copy the correct bbox directly from the region list.  Using pixel
# coords in the context but [0,1000] in the target would force the model to
# learn an implicit scale conversion, adding unnecessary difficulty.
#
# Output JSON schema rationale:
#   edit_type        -- allows Phase 3 to branch logic per operation
#   bbox             -- [x1,y1,x2,y2] in [0,1000] scale, copied from the
#                       segmentation region list that matches the instruction
#   edit_description -- echoes instruction; anchors spatial prediction to semantics

import json as _json_mod   # alias to avoid shadowing

PHASE1_SYSTEM_PROMPT = (
    "You are an image editing assistant. "
    "Given a source image, a list of segmentation regions with bounding boxes, "
    "and a natural-language edit instruction, "
    "identify the target region and predict the edit operation as a JSON object.\n\n"
    "The segmentation regions are listed as:\n"
    "  - <class_name>: [x1, y1, x2, y2]\n"
    "where coordinates are in [0, 1000] relative scale "
    "(0 = top-left, 1000 = bottom-right).\n\n"
    "JSON schema:\n"
    "{\n"
    '  "edit_type": "<type>",          '
    "// one of: action, add, adjust, background, content,\n"
    "//         hybrid, reference, remove, replace, style, version\n"
    '  "bbox": [x1, y1, x2, y2],       '
    "// the bounding box of the target region in [0, 1000] coordinates\n"
    "//   (select the region from the list above that best matches the instruction)\n"
    '  "edit_description": "<text>"    '
    "// precise description of what to edit\n"
    "}\n\n"
    "Output ONLY valid JSON. No explanation, no markdown code fences."
)

PHASE1_USER_REGIONS_HEADER = "Segmentation regions:\n"
PHASE1_USER_PREFIX          = "Edit instruction: "

# Hash the combined template so Phase 3 can detect drift at inference time.
import hashlib as _hashlib
_tpl_str = PHASE1_SYSTEM_PROMPT + "|SEP|" + PHASE1_USER_REGIONS_HEADER + "|SEP|" + PHASE1_USER_PREFIX
PHASE1_TEMPLATE_HASH = _hashlib.sha256(_tpl_str.encode()).hexdigest()[:16]
print(f"PHASE1_TEMPLATE_HASH = {repr(PHASE1_TEMPLATE_HASH)}")
print("  Save this value -- Phase 3 will verify it has not changed.")


def format_regions_text(annotations: list, seg_W: int, seg_H: int) -> str:
    '''Format all segmentation regions as the [0,1000] region list for the user turn.

    Args:
        annotations: list of annotation dicts with "class_name" and "bbox" fields.
        seg_W, seg_H: seg-file coordinate space dimensions for normalization.

    Returns:
        Multi-line string: "Segmentation regions:\n  - class: [x1,y1,x2,y2]\n..."
        Returns empty string if annotations is empty or None.

    Why [0,1000] in the prompt (not pixel coords):
        The model's output target is also [0,1000].  Showing regions in the
        same coordinate system lets the model learn to copy the correct bbox
        directly from the context, rather than learning an implicit scale
        conversion between context and output.

    Why include ALL annotations (not just object-region):
        The full scene context is what makes the spatial reasoning task
        non-trivial.  Background and other regions provide contrastive signal:
        the model must reject them when they do not match the instruction.
    '''
    if not annotations:
        return ""
    lines = [PHASE1_USER_REGIONS_HEADER]
    for ann in annotations:
        class_name = (ann.get("class_name") or "unknown").strip()
        raw_bbox   = ann.get("bbox") or []
        if len(raw_bbox) != 4:
            continue
        # Normalize to [0,1000] -- same scale as the output target JSON
        rel_bbox = bbox_to_relative(raw_bbox, seg_W, seg_H)
        lines.append(f"  - {class_name}: {rel_bbox}\n")
    return "".join(lines)


def build_messages(
    source_image,
    instruction_text: str,
    annotations: list = None,
    seg_dims: tuple   = None,
) -> list:
    '''Build Qwen2.5-VL chat messages for Phase 1 (user turn only; no assistant).

    Args:
        source_image    : PIL.Image.Image, or Path/str to image file.
        instruction_text: Natural-language edit instruction.
        annotations     : List of annotation dicts from the seg JSON.
                          Included as region context in the user turn.
                          If None or empty, the region list is omitted.
        seg_dims        : (seg_W, seg_H) -- coordinate space for bbox normalization.
                          Must match the space the annotation bboxes are stored in.
                          If None, defaults to (1, 1) -- do NOT rely on this default
                          when real annotations are present; always pass seg_dims.

    Returns:
        list of message dicts in Qwen2.5-VL format (system + user turns).

    INVARIANT: This is the ONLY function that constructs Phase 1 messages.
    Phase 3 inference must call this same function with the same arguments.
    Any change here invalidates the template hash and must bump PHASE1_TEMPLATE_HASH.

    User turn text layout:
        Segmentation regions:
          - shirt: [300, 400, 700, 900]
          - person: [100, 50, 900, 1200]
          ...
        Edit instruction: change the shirt to blue
    '''
    if isinstance(source_image, (str, Path)):
        source_image = Image.open(source_image).convert("RGB")
    elif not isinstance(source_image, Image.Image):
        raise TypeError(f"source_image must be PIL Image or Path, got {type(source_image)}")

    seg_W, seg_H = seg_dims if seg_dims else (1, 1)
    if seg_dims is None and annotations:
        # This is a programming error -- warn loudly rather than silently producing
        # wrong [0,1000] values (everything would collapse to [0,0,0,0]).
        log.warning(
            "build_messages: annotations provided but seg_dims is None. "
            "Region bboxes will be normalized to seg_dims=(1,1) -- WRONG. "
            "Always pass seg_dims when annotations are provided."
        )

    regions_text = format_regions_text(annotations or [], seg_W, seg_H)
    user_text    = regions_text + PHASE1_USER_PREFIX + instruction_text

    return [
        {"role": "system", "content": PHASE1_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": source_image},
                {"type": "text", "text": user_text},
            ],
        },
    ]


def build_target_json(edit_type: str, bbox: list, instruction_text: str) -> str:
    '''Build the assistant JSON response string for one training sample.

    Args:
        edit_type       : From manifest (one of KNOWN_EDIT_TYPES or "unknown").
        bbox            : [x1,y1,x2,y2] in [0,1000] scale from select_target_bbox().
        instruction_text: The original edit instruction (becomes edit_description).

    Returns:
        JSON string -- the exact text the model should learn to output.
    '''
    assert len(bbox) == 4, f"bbox must have 4 elements, got {len(bbox)}: {bbox}"
    assert edit_type in KNOWN_EDIT_TYPES or edit_type == "unknown", (
        f"Unexpected edit_type {edit_type!r}. "
        f"Known types: {sorted(KNOWN_EDIT_TYPES)}. "
        f"Add to expected_edit_types in CFG if intentional."
    )
    return _json_mod.dumps({
        "edit_type":        edit_type,
        "bbox":             [int(v) for v in bbox],
        "edit_description": instruction_text.strip(),
    }, ensure_ascii=False)


# -- Smoke tests -----------------------------------------------------------
_t = Image.new("RGB", (32, 32), (100, 100, 100))
_test_anns_msg = [
    {"class_name": "shirt",  "bbox": [300, 400, 700, 900]},
    {"class_name": "person", "bbox": [100, 50,  900, 1200]},
]
_m = build_messages(_t, "change the shirt to blue",
                    annotations=_test_anns_msg, seg_dims=(1920, 1080))
assert len(_m) == 2 and _m[0]["role"] == "system" and _m[1]["role"] == "user"
# Verify the user text contains the region list
_user_text = _m[1]["content"][1]["text"]
assert "shirt" in _user_text and "person" in _user_text, (
    f"Region list missing from user turn: {_user_text[:200]}"
)
assert PHASE1_USER_PREFIX in _user_text, "Edit instruction prefix missing from user turn"
assert "change the shirt to blue" in _user_text, "Instruction text missing from user turn"

# Test with no annotations (graceful degradation)
_m_no_ann = build_messages(_t, "Remove the tree")
assert PHASE1_USER_PREFIX in _m_no_ann[1]["content"][1]["text"]
assert "Segmentation regions" not in _m_no_ann[1]["content"][1]["text"]

_tj = build_target_json("adjust", [300, 400, 700, 900], "change the shirt to blue")
_p  = _json_mod.loads(_tj)
assert _p["edit_type"] == "adjust" and len(_p["bbox"]) == 4
print("  ok  build_messages and build_target_json validated (v1.3)")
print(f"\nExample user turn text (first 300 chars):")
print(_user_text[:300])
print(f"\nExample target JSON:\n{_tj}")

In [ ]:
# -- S8.2  build_vlm_dataset (v1.3) -------------------------------------------
# Loads samples_filtered.json, selects target bboxes via select_target_bbox(),
# and returns a list of raw training metadata dicts.  Images are NOT loaded here.
#
# v1.3 changes vs v1.2:
#   - route_bbox() replaced by select_target_bbox() (class-name matching)
#   - Each sample dict now includes "annotations" (the full annotation list)
#     and "seg_dims" (the seg-file resolution tuple).
#     These are propagated to tokenize_sample() and extract_hidden_state_for_sample()
#     so build_messages() receives the full scene context at every call site.
#   - Tracking of "matched_by" path (class_name / clip_score / area_fallback)
#     lets us audit how often class-name matching succeeds.
#
# Why store annotations in the sample dict (not re-load from disk each batch):
#   Annotation JSONs are small (~2-10 KB each).  Storing them avoids Drive
#   re-reads during training and hidden-state extraction.
#
# OOB audit: carried forward from v1.2 -- still checks bbox coords against
#   seg-file resolution before normalization to catch systematic misalignment.

OOB_WARN_FRAC  = 0.10
OOB_ERROR_FRAC = 0.50

from tqdm.auto import tqdm

def build_vlm_dataset(
    manifest_path: Path           = None,
    data_dir: Path                = None,
    sample_limit: Optional[int]   = None,
    warn_on_fallback: bool        = True,
) -> list:
    '''Build Phase 1 training metadata from samples_filtered.json.

    Returns:
        List of dicts with keys:
            sample_id         str   -- zero-padded 7-digit ID
            edit_type         str   -- confirmed edit type
            instruction       str   -- natural-language edit instruction
            bbox              list  -- [x1,y1,x2,y2] in [0,1000] from select_target_bbox()
            bbox_matched_by   str   -- selection path used (class_name/clip_score/area_fallback/global)
            target_json       str   -- JSON string for assistant response
            source_image_path Path  -- absolute path to source JPEG on Drive
            annotations       list  -- full annotation list from seg JSON (may be [])
            seg_dims          tuple -- (seg_W, seg_H) in seg-file coordinate space

    Raises:
        FileNotFoundError  if manifest is missing.
        AssertionError     if manifest has zero samples.
    '''
    if manifest_path is None:
        manifest_path = CFG.filtered_manifest_path
    if data_dir is None:
        data_dir = CFG.data_dir

    manifest = load_json(manifest_path)
    assert len(manifest) > 0, f"Manifest is empty: {manifest_path}"

    if sample_limit:
        manifest = manifest[:sample_limit]

    log.info(f"build_vlm_dataset: {len(manifest):,} samples from {manifest_path.name}")

    dataset      = []
    n_skipped    = 0
    n_global     = 0   # global-edit samples (full-image bbox, expected)
    n_class_match  = 0  # samples where class-name matched the instruction
    n_clip_break   = 0  # samples where clip_score broke a class-name tie
    n_area_fallback = 0 # samples where no class-name matched (warning path)
    n_oob        = 0
    n_with_bbox  = 0

    for entry in tqdm(manifest, desc="Building VLM dataset", unit="sample"):
        sid         = entry["sample_id"]
        edit_type   = entry.get("edit_type", "unknown")
        instruction = (entry.get("edit_instruction") or "").strip()
        W           = max(entry.get("img_width",  1) or 1, 1)
        H           = max(entry.get("img_height", 1) or 1, 1)
        src_rel     = entry.get("source_image", "")
        seg_rel     = entry.get("segmentation", "")

        if not src_rel:
            log.warning(f"Missing source_image in manifest for {sid} -- skipping")
            n_skipped += 1
            continue

        src_path = data_dir / src_rel
        if not src_path.exists():
            log.warning(f"Source image not on disk: {src_path} -- skipping")
            n_skipped += 1
            continue

        # -- Load segmentation JSON ------------------------------------------
        # seg_data["resolution"] is the coordinate space for bbox normalization.
        # Using manifest dims would produce wrong [0,1000] values when the seg
        # was generated at a different resolution than the stored JPEG.
        annotations = []
        seg_W, seg_H = W, H   # fallback: manifest dims if seg has no resolution
        if seg_rel:
            seg_path = data_dir / seg_rel
            try:
                if seg_path.exists() and seg_path.stat().st_size > 0:
                    seg_data    = load_json(seg_path)
                    annotations = seg_data.get("annotations", [])
                    _res = seg_data.get("resolution", {})
                    if _res.get("width") and _res.get("height"):
                        seg_W = int(_res["width"])
                        seg_H = int(_res["height"])
                        if seg_W != W or seg_H != H:
                            log.debug(
                                f"{sid}: seg resolution {seg_W}x{seg_H} differs from "
                                f"manifest {W}x{H} -- using seg dims for normalization"
                            )
                    else:
                        log.warning(
                            f"{sid}: seg file has no resolution field -- "
                            f"falling back to manifest dims {W}x{H}."
                        )
            except Exception as exc:
                log.warning(f"Seg JSON load error for {sid}: {exc} -- using empty annotations")

        # -- OOB audit (before normalization) --------------------------------
        _et_lower = (edit_type or "").strip().lower()
        _is_global = _et_lower in {"style", "background"}
        if not _is_global and _et_lower == "adjust":
            _is_global = any(kw in instruction.lower() for kw in GLOBAL_ADJUST_KEYWORDS)
        if not _is_global and annotations:
            _object_anns = [a for a in annotations if a.get("region") == "object"]
            _audit_anns  = _object_anns if _object_anns else annotations
            _best_raw, _best_area = None, 0.0
            for _ann in _audit_anns:
                _rb = _ann.get("bbox")
                if _rb and len(_rb) == 4:
                    _area = bbox_area_absolute(_rb)
                    if _area > _best_area:
                        _best_area = _area
                        _best_raw  = _rb
            if _best_raw is not None and _best_area > 0:
                n_with_bbox += 1
                if bbox_is_out_of_bounds(_best_raw, seg_W, seg_H):
                    n_oob += 1

        # -- Select target bbox via class-name matching ----------------------
        # select_target_bbox returns (bbox_rel, matched_by).
        # matched_by tells us which selection path was taken for this sample.
        bbox, matched_by = select_target_bbox(
            edit_type, instruction, annotations, seg_W, seg_H
        )

        FULL_IMAGE = [0, 0, 1000, 1000]
        if matched_by == "global":
            n_global += 1
        elif matched_by in ("class_name", "clip_score"):
            n_class_match += 1
            if matched_by == "clip_score":
                n_clip_break += 1
        else:  # area_fallback
            n_area_fallback += 1
            if warn_on_fallback:
                log.warning(
                    f"{sid}: no class-name match for instruction "
                    f"{instruction[:60]!r} -- used area fallback. "
                    "This sample has weaker supervision signal."
                )
            if bbox == FULL_IMAGE and not _is_global:
                log.warning(
                    f"{sid}: area fallback produced full-image bbox "
                    f"(edit_type={edit_type!r}) -- no valid annotations found. "
                    "Sample included but provides no spatial training signal."
                )

        target_json = build_target_json(edit_type, bbox, instruction)

        dataset.append({
            "sample_id":         sid,
            "edit_type":         edit_type,
            "instruction":       instruction,
            "bbox":              bbox,
            "bbox_matched_by":   matched_by,
            "target_json":       target_json,
            "source_image_path": src_path,
            # Full annotation list + seg dims for build_messages() at tokenize time.
            # Stored here (not re-loaded from disk) to avoid per-batch Drive reads.
            "annotations":       annotations,
            "seg_dims":          (seg_W, seg_H),
        })

    # -- OOB fraction check --------------------------------------------------
    if n_with_bbox > 0:
        oob_frac = n_oob / n_with_bbox
        _oob_msg = (
            f"{n_oob}/{n_with_bbox} localized-bbox samples "
            f"({oob_frac*100:.1f}%) had raw annotation coords outside image bounds."
        )
        if oob_frac >= OOB_ERROR_FRAC:
            log.error(f"DATASET ALIGNMENT ISSUE: {_oob_msg}")
        elif oob_frac >= OOB_WARN_FRAC:
            log.warning(f"Bbox OOB rate elevated: {_oob_msg}")
        else:
            log.info(f"Bbox OOB audit: {_oob_msg} (below warning threshold -- ok)")

    # -- Bbox selection path summary -----------------------------------------
    _total = len(dataset)
    log.info(
        f"build_vlm_dataset done: {_total:,} built, {n_skipped:,} skipped.\n"
        f"  Bbox selection breakdown:\n"
        f"    global (style/background)     : {n_global:,} ({n_global/_total*100:.1f}%)\n"
        f"    class-name matched            : {n_class_match:,} ({n_class_match/_total*100:.1f}%)\n"
        f"      of which clip_score tiebreak: {n_clip_break:,} ({n_clip_break/max(n_class_match,1)*100:.1f}% of matched)\n"
        f"    area fallback (no class match): {n_area_fallback:,} ({n_area_fallback/_total*100:.1f}%)"
    ) if _total > 0 else log.info("build_vlm_dataset: no samples built.")

    return dataset


print("build_vlm_dataset() defined (v1.3 -- class-name bbox selection).")
print(f"  OOB_WARN_FRAC  = {OOB_WARN_FRAC:.0%}")
print(f"  OOB_ERROR_FRAC = {OOB_ERROR_FRAC:.0%}")
print("  Each sample dict now includes: annotations, seg_dims, bbox_matched_by")


In [ ]:
# -- S8.3  Dataset smoke test -- verify 5 samples (no model loaded) -----------
# Run BEFORE S9 (model loading) to fail fast on data issues.
# Checks: manifest keys, image existence, bbox range, target JSON parseability,
#         annotations present, seg_dims set, bbox_matched_by recorded.

print("Building 5-sample test dataset...")
_smoke_raw = build_vlm_dataset(sample_limit=5, warn_on_fallback=True)

assert len(_smoke_raw) > 0, (
    "build_vlm_dataset returned 0 samples from samples_filtered.json. "
    "Verify Phase 0 S7.3 completed successfully and the manifest is non-empty."
)

print(f"\nok  {len(_smoke_raw)} samples built. Spot-checking structure:")
import json as _j

_required_keys = {
    "sample_id", "edit_type", "instruction", "bbox", "bbox_matched_by",
    "target_json", "source_image_path", "annotations", "seg_dims",
}

for i, s in enumerate(_smoke_raw):
    print(f"\n  Sample {i}: {s['sample_id']!r}")
    print(f"    edit_type      : {s['edit_type']!r}")
    print(f"    instruction    : {s['instruction'][:80]!r}")
    print(f"    bbox           : {s['bbox']}")
    print(f"    bbox_matched_by: {s['bbox_matched_by']!r}")
    print(f"    annotations    : {len(s['annotations'])} regions")
    print(f"    seg_dims       : {s['seg_dims']}")

    # Structural assertions
    _missing = _required_keys - set(s.keys())
    assert not _missing, f"Sample {i} missing keys: {_missing}"

    assert isinstance(s["bbox"], list) and len(s["bbox"]) == 4, \
        f"bbox must be list[4], got {s['bbox']}"
    assert all(0 <= v <= 1000 for v in s["bbox"]), \
        f"bbox values out of [0,1000]: {s['bbox']}"
    assert s["source_image_path"].exists(), \
        f"Source image missing: {s['source_image_path']}"
    assert isinstance(s["instruction"], str) and len(s["instruction"]) > 0
    assert s["bbox_matched_by"] in {"global","class_name","clip_score","area_fallback"}, \
        f"Unexpected bbox_matched_by: {s['bbox_matched_by']}"
    assert isinstance(s["annotations"], list), \
        f"annotations must be list, got {type(s['annotations'])}"
    assert isinstance(s["seg_dims"], tuple) and len(s["seg_dims"]) == 2, \
        f"seg_dims must be 2-tuple, got {s['seg_dims']}"

    # Target JSON must be parseable and have correct schema
    parsed = _j.loads(s["target_json"])
    assert "edit_type"        in parsed, "target_json missing edit_type"
    assert "bbox"             in parsed and len(parsed["bbox"]) == 4, "target_json bad bbox"
    assert "edit_description" in parsed, "target_json missing edit_description"
    assert all(0 <= v <= 1000 for v in parsed["bbox"]), \
        f"target_json bbox out of [0,1000]: {parsed['bbox']}"

    # If annotations exist, verify the region list would appear in the prompt
    if s["annotations"]:
        _regions_text = format_regions_text(s["annotations"], *s["seg_dims"])
        # At least one region line should be present
        assert "  - " in _regions_text, \
            f"format_regions_text produced no region lines for {s['sample_id']}"
        print(f"    sample region text preview: {_regions_text[:120]!r}")

print("\nok  All 5 samples passed structural assertions.")
_match_counts = {}
for s in _smoke_raw:
    _match_counts[s["bbox_matched_by"]] = _match_counts.get(s["bbox_matched_by"], 0) + 1
print(f"  Bbox selection path counts: {_match_counts}")

## S9 -- Phase 1: Model Loading (QLoRA)

Load Qwen2.5-VL-3B-Instruct with 4-bit NF4 quantization, attach LoRA adapters
to the language-model attention layers only, and prepare for gradient checkpointing.


In [ ]:
# -- S9.1  process_vision_info setup ------------------------------------------
# process_vision_info extracts PIL images from Qwen2.5-VL message dicts.
# Primary source: qwen_vl_utils (installed in S1.1).
# Fallback: inline implementation when qwen-vl-utils is unavailable.
# We always pass PIL Images in the "image" field -- no URL resolution needed.

try:
    from qwen_vl_utils import process_vision_info
    print("ok  Using qwen_vl_utils.process_vision_info")
except ImportError:
    print("WARN  qwen_vl_utils not available -- using built-in fallback")

    def process_vision_info(messages: list):
        '''Fallback: extract PIL images from Qwen2.5-VL message content dicts.

        Handles content items where "image" field is PIL.Image.Image or str path.
        Returns (image_list, video_list). video_list is always empty here.
        We always pass PIL images directly, so the str-path branch is a safety net.
        '''
        images = []
        for msg in messages:
            content = msg.get("content", [])
            if not isinstance(content, list):
                continue
            for item in content:
                if not isinstance(item, dict) or item.get("type") != "image":
                    continue
                img = item.get("image")
                if isinstance(img, Image.Image):
                    images.append(img)
                elif isinstance(img, (str, Path)):
                    images.append(Image.open(img).convert("RGB"))
                else:
                    raise TypeError(
                        f"Unsupported image type {type(img)} in message. "
                        "Pass PIL.Image.Image objects."
                    )
        return images, []   # (images, videos)

# Smoke-test
_pvi_img  = Image.new("RGB", (64, 64))
_pvi_msgs = [{"role": "user", "content": [{"type": "image", "image": _pvi_img}]}]
_imgs, _vids = process_vision_info(_pvi_msgs)
assert len(_imgs) == 1 and isinstance(_imgs[0], Image.Image)
assert _vids is None or len(_vids) == 0 # Changed assertion to handle NoneType
print("ok  process_vision_info smoke test passed")


In [ ]:
# -- S9.2  load_vlm_for_training -----------------------------------------------
# Loads Qwen2.5-VL-3B-Instruct with:
#   - 4-bit NF4 quantization (bitsandbytes==0.44.0)
#   - BF16 compute dtype
#   - LoRA adapters on LM attention modules only (NOT vision encoder)
#   - Gradient checkpointing enabled
#
# LoRA target modules:
#   Language model uses "q_proj", "k_proj", "v_proj", "o_proj" (separate).
#   Vision encoder uses "qkv" (fused QKV) -- these names are NOT matched below.
#   This ensures LoRA is attached only to the text transformer layers.
#   Verified at runtime in S9.3 -- do not assume, always print and check.

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training


def load_vlm_for_training(cfg=CFG):
    '''Load Qwen2.5-VL-3B with 4-bit QLoRA, ready for SFT.

    Returns:
        (model, processor) -- both on GPU with LoRA adapters attached.

    Verifies:
        - CFG.vlm_hidden_dim matches model.config.hidden_size
        - LoRA is NOT attached to any vision encoder module
        - LoRA IS attached to at least one language-model module
    '''
    bnb_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = "nf4",
        bnb_4bit_compute_dtype    = torch.bfloat16,
        bnb_4bit_use_double_quant = True,
    )
    print(f"Loading {cfg.vlm_model_id}  (4-bit NF4, BF16 compute) ...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        cfg.vlm_model_id,
        quantization_config = bnb_config,
        device_map          = "auto",
        torch_dtype         = torch.bfloat16,
    )
    model.config.use_cache = False   # required for gradient checkpointing

    # -- Verify hidden_dim ---------------------------------------------------
    actual_hidden = model.config.hidden_size
    if actual_hidden != cfg.vlm_hidden_dim:
        log.warning(
            f"CFG.vlm_hidden_dim={cfg.vlm_hidden_dim} but "
            f"model.config.hidden_size={actual_hidden}. "
            "Update CFG.vlm_hidden_dim to avoid Phase 2 shape mismatches."
        )
    else:
        print(f"  ok  hidden_dim={actual_hidden} matches CFG.vlm_hidden_dim")

    # -- Prepare 4-bit training (casts norms to FP32, enables input grads) --
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing       = True,
        gradient_checkpointing_kwargs    = {"use_reentrant": False},
    )

    # -- LoRA config ---------------------------------------------------------
    lora_config = LoraConfig(
        r              = cfg.vlm_lora_r,
        lora_alpha     = cfg.vlm_lora_alpha,
        lora_dropout   = cfg.vlm_lora_dropout,
        # Target ONLY language-model attention projections.
        # Qwen2.5-VL vision encoder uses fused "qkv" -- NOT matched here.
        # Qwen2.5-VL language model uses separate q/k/v/o projections.
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type      = TaskType.CAUSAL_LM,
        bias           = "none",
    )
    model = get_peft_model(model, lora_config)

    # -- Load processor ------------------------------------------------------
    processor = AutoProcessor.from_pretrained(cfg.vlm_model_id)
    # Qwen2.5-VL uses eos_token as pad by default; set explicitly for clarity
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
        print("  set pad_token = eos_token")

    # Cap image patch count so sequences stay within phase1_max_seq_len.
    # Default max_pixels for Qwen2.5-VL is ~12.8M → 2400+ image tokens.
    # 28*28*1024 = 802816 pixels → ≤1024 image tokens → seq fits in 2048.
    processor.image_processor.max_pixels = cfg.phase1_max_img_pixels
    processor.image_processor.min_pixels = 28 * 28 * 4
    print(f"  image_processor.max_pixels = {cfg.phase1_max_img_pixels}")

    return model, processor


print("load_vlm_for_training() defined.")
print(f"  Model   : {CFG.vlm_model_id}")
print(f"  Quant   : 4-bit NF4 + BF16 compute")
print(f"  LoRA r  : {CFG.vlm_lora_r}, alpha={CFG.vlm_lora_alpha}, dropout={CFG.vlm_lora_dropout}")


In [ ]:
# -- S9.3  Load model and verify LoRA attachment ------------------------------
# SPEC REQUIREMENT: "Do not assume model module names; print and verify them."
# This cell loads the model and asserts LoRA targets only LM layers.
# Expensive (~2-4 min on first run due to HuggingFace download).
# Guard against accidental reload of a fine-tuned model.

if "model" in dir() and model is not None:
    print("WARNING: 'model' variable already defined.")
    print("If you have a trained model in 'model', SAVE IT FIRST (S10.2).")
    print("Set model = None to proceed with reload.")

model, processor = load_vlm_for_training()

# -- Print all attention-related modules -------------------------------------
print("\n--- Attention/projection modules (sample) ---")
lm_proj_count  = 0
vis_proj_count = 0
lora_count     = 0

for name, module in model.named_modules():
    has_lora = hasattr(module, "lora_A")
    is_lm_proj = (
        any(name.endswith(f".{p}") for p in ["q_proj", "k_proj", "v_proj", "o_proj"])
        and "model.layers." in name
    )
    is_vis = "visual." in name or name.startswith("visual")

    if has_lora:
        lora_count += 1
    if is_lm_proj and not is_vis:
        lm_proj_count += 1
        if lm_proj_count <= 4:   # print first 4 LM proj modules
            lora_flag = " [LoRA]" if has_lora else ""
            print(f"  LM   {name}{lora_flag}")
    if is_vis and any(k in name for k in ["qkv", "proj"]) and vis_proj_count < 4:
        vis_proj_count += 1
        print(f"  VIS  {name}  (should have NO LoRA)")

print(f"\nLoRA modules total : {lora_count}")
print(f"LM proj modules    : {lm_proj_count}")
print(f"Vision proj modules: {vis_proj_count} (shown above, none should have LoRA)")

# -- Assertions --------------------------------------------------------------
assert lm_proj_count > 0, (
    "No LM attention projection modules found! "
    "Check model architecture -- expected 'model.layers.X.self_attn.*_proj'"
)
assert lora_count > 0, (
    "LoRA not attached to any module! get_peft_model may have failed. "
    "Check target_modules in LoraConfig match actual module names printed above."
)

# Verify NO vision encoder modules received LoRA
vis_with_lora = [
    n for n, m in model.named_modules()
    if hasattr(m, "lora_A") and ("visual." in n or n.startswith("visual"))
]
assert len(vis_with_lora) == 0, (
    f"LoRA accidentally attached to vision encoder: {vis_with_lora}. "
    "This wastes parameters and may degrade vision representations. "
    "Fix: ensure target_modules=['q_proj','k_proj','v_proj','o_proj'] only."
)

# -- Trainable parameter count -----------------------------------------------
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params     : {total:,}")
print(f"Trainable (LoRA) : {trainable:,}  ({trainable/total*100:.2f}%)")
assert 0 < trainable < total

print("\nok  LoRA verified: language-model layers only, vision encoder untouched.")


In [ ]:
# -- S9.4  tokenize_sample() + Phase1Dataset (v1.4) ----------------------------
#
# v1.4 debug changes vs v1.3:
#   1. os.path.exists() check BEFORE Image.open()  -- reason='missing_image'
#   2. Image.open() + .convert() wrapped in try/except -- reason='corrupt_image'
#   3. Every early-return now carries a reason string: {_skip:True, reason:'...'}
#      Reason codes:
#        missing_image        - source file absent from Drive
#        corrupt_image        - PIL cannot decode the file
#        exceeded_max_length  - seq_len > CFG.phase1_max_seq_len
#        no_assistant_start   - chat-template did not produce the expected role prefix
#        empty_response       - response_start >= seq_len (no response tokens)
#        exception:<type>     - unexpected exception during tokenization
#   4. Phase1Dataset propagates the reason so Phase1DataCollator can aggregate it.
#
# Everything else is identical to v1.3.

import os
import torch
from torch.utils.data import Dataset as TorchDataset

# The literal text that precedes the assistant response in Qwen2.5-VL chat format.
# Tokenised once at definition time and reused for all samples.
_ASSISTANT_ROLE_TEXT = "<|im_start|>assistant\n"


def tokenize_sample(
    sample_meta: dict,
    processor,
    max_length: int = None,
) -> dict:
    '''Tokenize one Phase 1 training sample.

    Returns:
        On success: dict with input_ids, attention_mask, labels, pixel_values,
                    image_grid_thw (all tensors).
        On failure: {"_skip": True, "reason": "<reason_code>"}
                    Never returns None -- always returns a dict so callers can
                    inspect the reason without isinstance/None checks.

    Reason codes: missing_image, corrupt_image:<type>, exceeded_max_length:<n>,
                  no_assistant_start, empty_response, exception:<type>
    '''
    if max_length is None:
        max_length = CFG.phase1_max_seq_len

    sample_id = sample_meta.get("sample_id", "?")

    # -- 1. Image existence check (fast, no I/O beyond stat) ---------------
    src_path = sample_meta["source_image_path"]
    if not os.path.exists(src_path):
        log.warning(
            f"tokenize_sample [{sample_id}]: image missing: {src_path}"
        )
        return {"_skip": True, "reason": "missing_image"}

    # -- 2. Image decode (isolated try/except for a specific reason code) --
    try:
        image = Image.open(src_path).convert("RGB")
    except Exception as e:
        log.warning(
            f"tokenize_sample [{sample_id}]: corrupt/unreadable image "
            f"({type(e).__name__}: {e}): {src_path}"
        )
        return {"_skip": True, "reason": f"corrupt_image:{type(e).__name__}"}

    # -- 3. Build messages and tokenize ------------------------------------
    try:
        # Build user-turn messages (canonical template from S8.1).
        # v1.3: pass annotations + seg_dims so the region list is included.
        messages = build_messages(
            image,
            sample_meta["instruction"],
            annotations = sample_meta.get("annotations") or [],
            seg_dims    = sample_meta.get("seg_dims"),
        )

        # Full conversation including assistant response
        full_messages = messages + [
            {"role": "assistant", "content": sample_meta["target_json"]}
        ]

        # Apply chat template -- returns text with <|image_pad|> placeholders
        full_text = processor.apply_chat_template(
            full_messages, tokenize=False, add_generation_prompt=False
        )

        # Extract PIL images for processor (uses build_messages output)
        image_inputs, video_inputs = process_vision_info(messages)

        # Tokenize + encode image patches into pixel_values
        inputs = processor(
            text          = [full_text],
            images        = image_inputs if image_inputs else None,
            videos        = video_inputs if video_inputs else None,
            padding       = False,
            return_tensors = "pt",
        )

        input_ids      = inputs["input_ids"][0]      # (seq_len,)
        attention_mask = inputs["attention_mask"][0]  # (seq_len,)
        seq_len        = input_ids.shape[0]

        # -- 4. Max-length gate -------------------------------------------
        if seq_len > max_length:
            log.debug(
                f"tokenize_sample [{sample_id}]: "
                f"seq_len={seq_len} > max_length={max_length} -- skip"
            )
            return {"_skip": True, "reason": f"exceeded_max_length:{seq_len}"}

        # -- 5. Build labels: -100 for all non-response tokens ------------
        # Find LAST occurrence of <|im_start|>assistant\n in input_ids.
        # Searching from the END handles the pathological case where the
        # instruction text contains the same byte sequence.
        assistant_start_ids = processor.tokenizer.encode(
            _ASSISTANT_ROLE_TEXT, add_special_tokens=False
        )
        k = len(assistant_start_ids)

        response_start = None
        id_list = input_ids.tolist()
        for i in range(seq_len - k, -1, -1):
            if id_list[i : i + k] == assistant_start_ids:
                response_start = i + k   # first token of the actual JSON response
                break

        if response_start is None:
            log.warning(
                f"tokenize_sample [{sample_id}]: cannot find assistant role tokens. "
                f"Chat template may have changed. "
                f"_ASSISTANT_ROLE_TEXT={_ASSISTANT_ROLE_TEXT!r}"
            )
            return {"_skip": True, "reason": "no_assistant_start"}

        if response_start >= seq_len:
            log.warning(
                f"tokenize_sample [{sample_id}]: "
                f"response_start={response_start} >= seq_len={seq_len} -- empty response"
            )
            return {"_skip": True, "reason": "empty_response"}

        labels = torch.full((seq_len,), -100, dtype=torch.long)
        labels[response_start:] = input_ids[response_start:]

        # Invariant: at least 1 non-masked label token
        n_response = (labels != -100).sum().item()
        assert n_response > 0, (
            f"All labels are -100 for [{sample_id}]. "
            f"response_start={response_start}, seq_len={seq_len}"
        )

        return {
            "input_ids":      input_ids.long(),
            "attention_mask": attention_mask.long(),
            "labels":         labels.long(),
            "pixel_values":   inputs.get("pixel_values"),     # (n_patches, patch_dim)
            "image_grid_thw": inputs.get("image_grid_thw"),   # (n_images, 3)
        }

    except Exception as e:
        log.warning(
            f"tokenize_sample [{sample_id}]: "
            f"unexpected exception ({type(e).__name__}: {e})"
        )
        return {"_skip": True, "reason": f"exception:{type(e).__name__}"}


class Phase1Dataset(TorchDataset):
    '''Lazy-loading torch Dataset for Phase 1 VLM fine-tuning.

    Loads images from Drive and tokenizes in __getitem__. Avoids holding
    all pixel_values in memory simultaneously.

    DataLoader note: use dataloader_num_workers=0 to avoid PIL/multiprocessing
    issues with the Qwen2.5-VL processor in Colab.
    '''

    def __init__(self, raw_samples: list, processor, max_length: int = None):
        self.samples    = raw_samples
        self.processor  = processor
        self.max_length = max_length or CFG.phase1_max_seq_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        result = tokenize_sample(self.samples[idx], self.processor, self.max_length)
        # tokenize_sample always returns a dict now (never None).
        # Pass the reason through so Phase1DataCollator can aggregate it.
        if result.get("_skip"):
            return {"_skip": True, "reason": result.get("reason", "unknown")}
        return result


print("tokenize_sample() and Phase1Dataset defined (v1.4).")
print(f"  max_length            = {CFG.phase1_max_seq_len}")
print(f"  _ASSISTANT_ROLE_TEXT  = {_ASSISTANT_ROLE_TEXT!r}")
print("  Skip reasons tracked  : missing_image, corrupt_image, exceeded_max_length,")
print("                          no_assistant_start, empty_response, exception:<type>")


In [ ]:
# -- S9.5  Phase1DataCollator (v1.4) ------------------------------------------
#
# v1.4 debug changes vs v1.3:
#   1. self._call_count tracks the batch index so crash messages are pinpointed.
#   2. On empty-batch: aggregates skip reasons from the _skip dicts and prints a
#      breakdown instead of a generic error string.
#   3. batch_idx is included in the RuntimeError so log tails show which batch failed.
#
# Everything else is identical to v1.3.

from collections import Counter as _Counter

class Phase1DataCollator:
    '''DataCollator for Qwen2.5-VL Phase 1 SFT training.

    Filters failed samples, dynamically pads text sequences, and concatenates
    image patches into the format expected by Qwen2.5-VL forward().
    '''

    def __init__(self, processor):
        self.pad_id = (
            processor.tokenizer.pad_token_id
            if processor.tokenizer.pad_token_id is not None
            else 0
        )
        # Counts how many batches have been collated.
        # Used to identify the failing batch index in crash messages.
        self._call_count = 0

    def __call__(self, features: list) -> dict:
        self._call_count += 1
        batch_idx = self._call_count   # 1-indexed for human readability

        # -- Collect skip reasons BEFORE filtering (for diagnostics) --------
        skipped = [f for f in features if not f or f.get("_skip", False)]
        skip_reasons = [f.get("reason", "unknown") for f in skipped if f]

        # -- Filter failed / sentinel samples --------------------------------
        features = [f for f in features if f and not f.get("_skip", False)]

        if len(features) == 0:
            reason_counts = _Counter(skip_reasons)
            reason_str = "  |  ".join(
                f"{r}: {n}" for r, n in reason_counts.most_common()
            ) or "no reasons recorded"
            raise RuntimeError(
                f"Empty batch at batch_idx={batch_idx} after filtering "
                f"{len(skipped)} failed sample(s).\n"
                f"  Skip reason breakdown: {reason_str}\n"
                f"  Fix: run S9.5b audit cell to pre-filter the dataset before "
                f"training, or increase CFG.phase1_max_seq_len."
            )

        if skipped:
            log.debug(
                f"Collator batch_idx={batch_idx}: "
                f"{len(skipped)} sample(s) skipped "
                f"({dict(_Counter(skip_reasons))}), "
                f"{len(features)} sample(s) remain."
            )

        # -- Determine max sequence length in this batch ---------------------
        max_len = max(f["input_ids"].shape[0] for f in features)

        padded_input_ids = []
        padded_attn_mask = []
        padded_labels    = []

        for f in features:
            seq_len = f["input_ids"].shape[0]
            pad_len = max_len - seq_len

            padded_input_ids.append(
                torch.cat([f["input_ids"],
                           torch.full((pad_len,), self.pad_id, dtype=torch.long)])
            )
            padded_attn_mask.append(
                torch.cat([f["attention_mask"],
                           torch.zeros(pad_len, dtype=torch.long)])
            )
            padded_labels.append(
                torch.cat([f["labels"],
                           torch.full((pad_len,), -100, dtype=torch.long)])
            )

        result = {
            "input_ids":      torch.stack(padded_input_ids),   # (B, max_len)
            "attention_mask": torch.stack(padded_attn_mask),   # (B, max_len)
            "labels":         torch.stack(padded_labels),      # (B, max_len)
        }

        # -- Pixel values: concatenate patches across batch ------------------
        # Each sample: pixel_values=(n_patches_i, patch_dim), image_grid_thw=(1,3)
        # Batch result: pixel_values=(sum_n_patches, patch_dim), image_grid_thw=(B,3)
        pvs   = [f["pixel_values"]   for f in features if f.get("pixel_values")   is not None]
        grids = [f["image_grid_thw"] for f in features if f.get("image_grid_thw") is not None]

        if pvs:
            result["pixel_values"]   = torch.cat(pvs,   dim=0)
            result["image_grid_thw"] = torch.cat(grids, dim=0)

        return result


print("Phase1DataCollator defined (v1.4).")
print("  Handles: sentinel filtering, right-padding, pixel_value concatenation")
print("  New: batch_idx logged on error, skip-reason breakdown on empty batch")
print("  Remember: set remove_unused_columns=False in TrainingArguments (see S10.1)")


## S9.5b -- Pre-Training Dataset Audit

Run `audit_tokenization()` across **all** samples before training starts.

- Iterates every sample through `tokenize_sample` with the real processor.
- Logs **min / max / mean sequence length** across the first 100 valid samples -- confirms whether `CFG.phase1_max_seq_len` is actually sufficient.
- Prints a **failure-reason breakdown** so you can see exactly what proportion of samples fail and why (missing image, corrupt image, exceeded max length, etc.).
- Filters `_full_raw` to only the passing indices so the collator can never receive an all-skip batch during training.

**Run this cell after model load (S9.3) but before S10.1 training.**

In [ ]:
# -- S9.5b  Pre-training dataset audit ----------------------------------------
#
# WHY: The 'Empty batch after filtering failed samples' error fires when EVERY
#   sample in at least one DataLoader batch fails tokenization.  Rather than
#   discovering this mid-training, we audit the full dataset first, collect
#   failure reasons, and filter to only valid samples before training starts.
#
# WHAT THIS CELL DOES:
#   1. Runs tokenize_sample on every sample in _full_raw.
#   2. Logs token length stats (min/max/mean/median) for the first 100 valid
#      samples so you can verify CFG.phase1_max_seq_len is set appropriately.
#   3. Prints a failure-reason breakdown with counts and percentages.
#   4. Filters _full_raw to _full_raw_filtered (passing samples only).
#   5. S10.1 builds Phase1Dataset from _full_raw_filtered.
#
# RESTART SAFETY: re-running is safe -- no Drive writes happen here.
#
# RUNTIME: ~1-2 ms per sample (image load + processor encode).
#   10k samples is roughly 2-4 minutes on an A100.

import os
import statistics
from collections import Counter
from tqdm.auto import tqdm


def audit_tokenization(
    raw_samples: list,
    processor,
    max_length: int = None,
    log_token_lengths_n: int = 100,
) -> tuple:
    '''Run tokenize_sample on every sample; collect pass/fail stats.

    Args:
        raw_samples:         list of dicts from build_vlm_dataset()
        processor:           AutoProcessor for Qwen2.5-VL
        max_length:          passed through to tokenize_sample (None uses CFG default)
        log_token_lengths_n: number of valid samples to measure seq lengths over

    Returns:
        (good_indices, fail_reasons) where:
            good_indices: list[int] -- indices into raw_samples that tokenized OK
            fail_reasons: list[(int, str)] -- (idx, reason) for each skipped sample
    '''
    if max_length is None:
        max_length = CFG.phase1_max_seq_len

    good_indices = []
    fail_reasons = []
    seq_lengths  = []   # seq lengths of first log_token_lengths_n valid samples

    print(f"Auditing {len(raw_samples):,} samples through tokenize_sample ...")
    print(f"  CFG.phase1_max_seq_len = {max_length}")
    print()

    for idx, sample in enumerate(tqdm(raw_samples, desc="Audit", unit="sample")):
        result = tokenize_sample(sample, processor, max_length)

        if result.get("_skip"):
            reason = result.get("reason", "unknown")
            fail_reasons.append((idx, reason))
        else:
            good_indices.append(idx)
            if len(seq_lengths) < log_token_lengths_n:
                seq_lengths.append(result["input_ids"].shape[0])

    # -- Token length stats -----------------------------------------------
    print(f"\n--- Token length stats (first {len(seq_lengths)} valid samples) ---")
    if seq_lengths:
        print(f"  min    : {min(seq_lengths)}")
        print(f"  max    : {max(seq_lengths)}")
        print(f"  mean   : {statistics.mean(seq_lengths):.1f}")
        print(f"  median : {statistics.median(seq_lengths):.1f}")
        n_over = sum(1 for l in seq_lengths if l > max_length)
        pct_over = 100 * n_over / len(seq_lengths)
        print(f"  > {max_length} (max_length) : {n_over}/{len(seq_lengths)}  ({pct_over:.1f}%)")
        if pct_over > 50:
            print(f"  WARN: >50% of sampled sequences exceed max_length={max_length}. ")
            print(f"        Consider increasing CFG.phase1_max_seq_len.")
    else:
        print("  WARN: No valid samples to measure. All samples may be failing.")

    # -- Failure breakdown ------------------------------------------------
    n_total = len(raw_samples)
    n_good  = len(good_indices)
    n_fail  = len(fail_reasons)

    print(f"\n--- Audit summary ---")
    print(f"  Total    : {n_total:,}")
    print(f"  Passed   : {n_good:,}  ({n_good / n_total * 100:.1f}%)")
    print(f"  Failed   : {n_fail:,}  ({n_fail / n_total * 100:.1f}%)")

    if fail_reasons:
        reason_counts = Counter(r for _, r in fail_reasons)
        print(f"\n  Failure breakdown:")
        for reason, count in reason_counts.most_common():
            pct = 100 * count / n_total
            print(f"    {count:>6,}  ({pct:5.1f}%)  {reason}")
        # Show first 5 sample IDs for the most common failure reason
        top_reason = reason_counts.most_common(1)[0][0]
        examples = [
            raw_samples[i]["sample_id"]
            for i, r in fail_reasons if r == top_reason
        ][:5]
        print(f"\n  First failing sample_ids for '{top_reason}': {examples}")

    if n_good == 0:
        raise RuntimeError(
            "Audit found ZERO valid samples. Training would immediately fail.\n"
            "Review the failure breakdown above and fix the root cause.\n"
            "Common fixes:\n"
            "  missing_image        : verify Drive mount and CFG.data_dir\n"
            "  exceeded_max_length  : increase CFG.phase1_max_seq_len\n"
            "  corrupt_image        : re-download the dataset (Phase 0)\n"
        )

    return good_indices, fail_reasons


# -- Load _full_raw if not already in scope --------------------------------
# S9.5b can be re-run after a kernel restart: only needs processor and
# the saved vlm_dataset_phase1.json from cell S8.4.
if "_full_raw" not in dir() or _full_raw is None:
    print("_full_raw not in scope -- loading from vlm_dataset_phase1.json ...")
    _full_raw = load_vlm_dataset(CFG.data_dir / "vlm_dataset_phase1.json")

# -- Run the audit --------------------------------------------------------
_good_indices, _fail_reasons = audit_tokenization(
    raw_samples         = _full_raw,
    processor           = processor,
    max_length          = CFG.phase1_max_seq_len,
    log_token_lengths_n = 100,
)

# -- Filter to passing samples only ---------------------------------------
_full_raw_filtered = [_full_raw[i] for i in _good_indices]

print(f"\nFiltered dataset: {len(_full_raw):,} -> {len(_full_raw_filtered):,} samples")
print(f"({len(_fail_reasons):,} samples removed before training)")
print("_full_raw_filtered is ready for S10.1.")


In [ ]:
# Define _full_raw for this smoke test if it hasn't been defined by S10.1 yet.
# This makes S9.6 self-contained for shape validation.
if '_full_raw' not in locals() or _full_raw is None:
    print("  _full_raw not found, building a small test dataset...")
    _full_raw = build_vlm_dataset(
        manifest_path=CFG.filtered_manifest_path,
        data_dir=CFG.data_dir,
        sample_limit=10, # Enough samples to get 2 valid ones for the test
        warn_on_fallback=False,
    )
    if len(_full_raw) == 0:
        raise RuntimeError("build_vlm_dataset returned 0 samples for smoke test. Check manifest.")


# ── DIAGNOSTIC: check actual sequence lengths before max_length gate ──────
# Run this cell before S9.6 to confirm the root cause.

print("Diagnosing tokenize_sample failures...")
print(f"  CFG.phase1_max_seq_len in memory = {CFG.phase1_max_seq_len}")
print()

_diag_raw = _full_raw[:4] if '_full_raw' in locals() else build_vlm_dataset(
    manifest_path=CFG.filtered_manifest_path, data_dir=CFG.data_dir, sample_limit=4
)

from qwen_vl_utils import process_vision_info

for _i, _s in enumerate(_diag_raw[:4]):
    try:
        _img = Image.open(_s["source_image_path"]).convert("RGB")
        _msgs = build_messages(
            _img, _s["instruction"],
            annotations=_s.get("annotations") or [],
            seg_dims=_s.get("seg_dims"),
        )
        _full_msgs = _msgs + [{"role": "assistant", "content": _s["target_json"]}]
        _full_text = processor.apply_chat_template(
            _full_msgs, tokenize=False, add_generation_prompt=False
        )
        _img_inputs, _vid_inputs = process_vision_info(_msgs)
        _enc = processor(
            text=[_full_text],
            images=_img_inputs if _img_inputs else None,
            videos=_vid_inputs if _vid_inputs else None,
            padding=False, return_tensors="pt",
        )
        _slen = _enc["input_ids"].shape[1]
        _status = "OK" if _slen <= CFG.phase1_max_seq_len else f"EXCEEDS_LIMIT (limit={CFG.phase1_max_seq_len})"
        print(f"  sample {_i} ({_s['sample_id'][:20]}): seq_len={_slen}  [{_status}]")
    except Exception as _e:
        print(f"  sample {_i}: EXCEPTION → {type(_e).__name__}: {_e}")

In [ ]:
# -- S9.6  Shape validation smoke test ----------------------------------------
# SPEC REQUIREMENT: "Before writing any training loop, ensure you can run one
# forward pass through the [model] with correct shapes."
# This cell is the mandatory go/no-go gate before S10 training.
#
# Asserts at every bridge point:
#   - input_ids shape (B, seq_len)
#   - attention_mask shape == input_ids shape
#   - labels shape == input_ids shape, with >=1 non-(-100) token per sample
#   - pixel_values ndim >= 2
#   - image_grid_thw shape[0] == B (one grid per image, one image per sample)
#   - forward pass produces non-NaN loss
#   - logits shape (B, seq_len, vocab_size)

print("Building shape-validation batch (2 valid samples from first 8)...")

# Define _full_raw for this smoke test if it hasn't been defined by S10.1 yet.
# This makes S9.6 self-contained for shape validation.
if '_full_raw' not in locals() or _full_raw is None:
    print("  _full_raw not found, building a small test dataset...")
    _full_raw = build_vlm_dataset(
        manifest_path=CFG.filtered_manifest_path,
        data_dir=CFG.data_dir,
        sample_limit=10, # Enough samples to get 2 valid ones for the test
        warn_on_fallback=False,
    )
    if len(_full_raw) == 0:
        raise RuntimeError("build_vlm_dataset returned 0 samples for smoke test. Check manifest.")

_val_ds  = Phase1Dataset(_full_raw[:8], processor)
_val_col = Phase1DataCollator(processor)

# Collect 2 valid tokenized samples (skip any that return _skip sentinel)
_val_items = []
for _i in range(len(_val_ds)):
    _item = _val_ds[_i]
    if not _item.get("_skip"):
        _val_items.append(_item)
    if len(_val_items) == 2:
        break

if len(_val_items) < 2:
    raise RuntimeError(
        f"Only {len(_val_items)} valid samples from first 8 entries. "
        "Check tokenize_sample logs. "
        "Possible causes: all samples exceed max_seq_len or images are corrupt."
    )

_val_batch = _val_col(_val_items)
B = len(_val_items)

print(f"\n--- Batch tensor shapes (B={B}) ---")
for k, v in _val_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<20s}: {tuple(v.shape)}  dtype={v.dtype}")

# -- Shape assertions --------------------------------------------------------
assert _val_batch["input_ids"].shape[0]      == B,  "input_ids batch dim"
assert _val_batch["attention_mask"].shape    == _val_batch["input_ids"].shape, \
    "attention_mask shape mismatch"
assert _val_batch["labels"].shape            == _val_batch["input_ids"].shape, \
    "labels shape mismatch"

for b in range(B):
    n_resp = (_val_batch["labels"][b] != -100).sum().item()
    assert n_resp > 0, f"Sample {b} has all-masked labels (response_start bug?)"
    seq_len = _val_batch["input_ids"].shape[1]
    print(f"  Sample {b}: {n_resp} response tokens / {seq_len} total seq_len")

if "pixel_values" in _val_batch:
    pv = _val_batch["pixel_values"]
    assert pv.ndim >= 2, f"Expected >=2D pixel_values, got {pv.shape}"
    gthw = _val_batch["image_grid_thw"]
    assert gthw.shape[0] == B, (
        f"image_grid_thw.shape[0]={gthw.shape[0]} != B={B}. "
        "Each sample should have exactly 1 source image."
    )
    print(f"\n  pixel_values    : {tuple(pv.shape)}")
    print(f"  image_grid_thw  : {tuple(gthw.shape)}")

# -- Run forward pass --------------------------------------------------------
print("\nRunning forward pass (no_grad, eval mode)...")
model.eval()
_dev = str(next(model.parameters()).device)
_b_dev = {k: v.to(_dev) if isinstance(v, torch.Tensor) else v
          for k, v in _val_batch.items()}

with torch.no_grad():
    _out = model(**_b_dev)

# Assert output shapes
assert _out.loss is not None, "Model returned no loss -- check labels tensor"
assert not _out.loss.isnan().item(), (
    f"NaN loss in forward pass! "
    f"Possible causes: NaN in pixel_values, all-(-100) labels, mixed precision issue."
)
assert _out.logits.shape == (B, _val_batch["input_ids"].shape[1], model.config.vocab_size), \
    f"Unexpected logits shape: {_out.logits.shape}"

print(f"\n  loss   : {_out.loss.item():.4f}")
print(f"  logits : {tuple(_out.logits.shape)}")
print(f"\nok  Shape validation passed. Loss={_out.loss.item():.4f}")
print("    All tensor shapes correct. Ready for S10 training.")
model.train()


In [ ]:
import json
from pathlib import Path

def save_vlm_dataset(dataset: list, output_path: Path) -> None:
    """Serialize build_vlm_dataset() output to JSON on Drive.

    Converts Path -> str and tuple -> list for JSON compatibility.
    Safe to re-run: overwrites existing file.
    """
    serializable = []
    for s in dataset:
        serializable.append({
            **s,
            "source_image_path": str(s["source_image_path"]),
            "seg_dims":          list(s["seg_dims"]),
        })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(serializable, indent=2, ensure_ascii=False))
    size_mb = output_path.stat().st_size / 1e6
    print(f"Saved {len(dataset):,} samples → {output_path} ({size_mb:.1f} MB)")


def load_vlm_dataset(input_path: Path) -> list:
    """Load a dataset saved by save_vlm_dataset().

    Restores source_image_path -> Path and seg_dims -> tuple.
    """
    raw = json.loads(input_path.read_text())
    dataset = []
    for s in raw:
        dataset.append({
            **s,
            "source_image_path": Path(s["source_image_path"]),
            "seg_dims":          tuple(s["seg_dims"]),
        })
    print(f"Loaded {len(dataset):,} samples from {input_path}")
    return dataset

In [ ]:
vlm_train = build_vlm_dataset()
save_vlm_dataset(vlm_train, CFG.data_dir / "vlm_dataset_phase1.json")

# Section 10 -- Phase 1 Training (SFTTrainer + QLoRA)

This section executes the Qwen2.5-VL-3B QLoRA fine-tuning run.

**Design decisions:**
- `SFTTrainer` from TRL 0.11.0 is used with a **pre-tokenized** `Phase1Dataset`, not `formatting_func`.
  Using `formatting_func` causes TRL to re-tokenize internally and silently drop `pixel_values`
  because `DataCollatorForSeq2Seq` does not handle multimodal fields. Pre-tokenized datasets
  bypass this path entirely.
- `remove_unused_columns=False` is mandatory. Without it, `Trainer` inspects the model's
  `forward()` signature and drops any batch key not in the signature -- including `pixel_values`
  and `image_grid_thw`, which are passed through `**kwargs` in Qwen2.5-VL and not listed
  explicitly in the Python signature.
- Checkpoints are saved to Drive (`CFG.checkpoint_dir`) so training can resume after a
  Colab disconnect without restarting from epoch 0.
- `paged_adamw_8bit` reduces optimizer state memory by ~75% vs AdamW, essential at 3B param
  scale with 4-bit weights.
- `dataloader_num_workers=0`: PIL image loading inside `__getitem__` is not fork-safe on
  Colab; multiprocessing workers cause hang or corrupted image reads.


In [ ]:
# -- S10.1  SFTTrainer training setup and execution ----------------------------
# SPEC REQUIREMENT: Phase 1 trains Qwen2.5-VL-3B with QLoRA for structured
# JSON edit prediction. Training uses the full filtered manifest.
#
# CRITICAL FLAGS:
#   remove_unused_columns=False  -- prevents Trainer from dropping pixel_values
#   dataloader_num_workers=0     -- PIL not fork-safe in Colab subprocesses
#   dataset_text_field=None      -- suppresses SFTTrainer's default text wrapping
#   packing=False                -- packing not supported with image inputs
#   paged_adamw_8bit             -- essential for memory at 3B scale
#
# Restart safety:
#   TrainingArguments.resume_from_checkpoint is set automatically if a
#   checkpoint directory already exists in CFG.checkpoint_dir.
#   New checkpoints overwrite the oldest when save_total_limit is set.

import os, glob as _glob_mod
from transformers import TrainingArguments
from trl import SFTTrainer

# -- Build full training dataset ----------------------------------------------
#print("Building Phase1Dataset from full filtered manifest...")
#_full_raw = build_vlm_dataset(
#    manifest_path  = CFG.filtered_manifest_path,
#    data_dir       = CFG.data_dir,
#    sample_limit   = None,          # all samples
#    warn_on_fallback = True,
#)
#print(f"  Total training samples: {len(_full_raw)}")
#assert len(_full_raw) > 0, "Empty training set -- check filtered_manifest_path and build_vlm_dataset"

# -- Use pre-audited filtered dataset from S9.5b if available. ---------------
# S9.5b removes samples that fail tokenization (missing images, corrupt files,
# sequences that exceed max_length) so the collator never sees an all-skip batch.
# If S9.5b was not run, fall back to the unfiltered dataset with a warning.
if "_full_raw_filtered" in dir() and _full_raw_filtered:
    print(f"  Using pre-filtered dataset from S9.5b: {len(_full_raw_filtered):,} samples")
    _train_raw = _full_raw_filtered
else:
    print("  WARNING: _full_raw_filtered not found -- training without pre-filtering.")
    print("  Run S9.5b audit cell first to avoid 'Empty batch' crashes.")
    if "_full_raw" not in dir() or _full_raw is None:
        _full_raw = load_vlm_dataset(CFG.data_dir / "vlm_dataset_phase1.json")
    _train_raw = _full_raw

train_dataset = Phase1Dataset(_train_raw, processor)

# -- Detect existing checkpoint for resume ------------------------------------
def _latest_checkpoint(ckpt_dir: str):
    '''Return path to latest checkpoint in ckpt_dir, or None.'''
    pattern = os.path.join(ckpt_dir, "checkpoint-*")
    ckpts   = sorted(_glob_mod.glob(pattern),
                     key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else 0)
    return ckpts[-1] if ckpts else None


_ckpt_dir  = str(CFG.ckpt_phase1)
_resume    = _latest_checkpoint(_ckpt_dir)
if _resume:
    print(f"  Resuming from checkpoint: {_resume}")
else:
    print(f"  No existing checkpoint found. Starting fresh training.")
    CFG.ckpt_phase1.mkdir(parents=True, exist_ok=True)

# -- TrainingArguments --------------------------------------------------------
training_args = TrainingArguments(
    output_dir                  = _ckpt_dir,
    num_train_epochs            = CFG.phase1_epochs,
    per_device_train_batch_size = CFG.phase1_batch_size,
    gradient_accumulation_steps = CFG.phase1_grad_accum,
    learning_rate               = CFG.phase1_lr,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    optim                       = "paged_adamw_8bit",
    bf16                        = True,
    fp16                        = False,
    logging_steps               = 10,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    # CRITICAL: without this, Trainer drops pixel_values before calling collator
    remove_unused_columns       = False,
    # PIL image loading is not fork-safe in Colab; keep at 0
    dataloader_num_workers      = 0,
    report_to                   = "none",
    run_name                    = f"phase1-vlm-{CFG.phase1_epochs}ep",
    # gradient checkpointing already enabled in model via prepare_model_for_kbit_training
    gradient_checkpointing      = False,  # set on model directly, not here
)

# -- SFTTrainer ---------------------------------------------------------------
# dataset_text_field=None: we supply pre-tokenized tensors, not raw text strings.
# SFTTrainer will use input_ids/labels from the dataset and skip its own tokenization.
# data_collator=Phase1DataCollator: handles pixel_values concatenation + sentinel filtering.
trainer = SFTTrainer(
    model             = model,
    args              = training_args,
    train_dataset     = train_dataset,
    data_collator     = Phase1DataCollator(processor),
    dataset_text_field = None,  # suppress default text tokenization path
    packing           = False,  # not supported with image inputs
    max_seq_length    = CFG.phase1_max_seq_len,
)

# -- Sanity: confirm remove_unused_columns made it through -------------------
assert trainer.args.remove_unused_columns == False, (
    "remove_unused_columns was reset to True. "
    "This will cause pixel_values to be dropped during training. "
    "Check TRL version -- SFTTrainer in older versions forced this to True."
)

print("\n--- Training configuration ---")
print(f"  Epochs            : {CFG.phase1_epochs}")
print(f"  Batch size        : {CFG.phase1_batch_size}  x  grad_accum={CFG.phase1_grad_accum}")
print(f"  Effective batch   : {CFG.phase1_batch_size * CFG.phase1_grad_accum}")
print(f"  Learning rate     : {CFG.phase1_lr}")
print(f"  Optimizer         : paged_adamw_8bit")
print(f"  Checkpoint dir    : {_ckpt_dir}")
print(f"  Resume from       : {_resume or 'scratch'}")
print(f"  remove_unused_columns : False (verified)")
print(f"  Training samples  : {len(train_dataset)}")

# -- Execute training ---------------------------------------------------------
print("\nStarting training... (this will take 1-3 hours on a T4/A100)")
trainer.train(resume_from_checkpoint=_resume)

print("\nok  Training complete.")
print(f"    Final checkpoint saved to: {_ckpt_dir}")


In [ ]:
# -- S10.2  Save final LoRA adapter to Drive ----------------------------------
# Saves ONLY the LoRA adapter weights (not the frozen base model quantized
# weights, which are already on Drive in the HuggingFace cache).
# This keeps the save small (~60-120 MB for r=16 on 3B model).
#
# Also saves:
#   - processor (tokenizer + image processor config)
#   - PHASE1_TEMPLATE_HASH as adapter_config metadata for Phase 3 drift detection
#
# Why save_pretrained with safe_serialization=True:
#   safetensors format is faster to load and immune to pickle deserialization bugs.
#
# The adapter can be reloaded with:
#   model = PeftModel.from_pretrained(base_model, adapter_save_path)

import json as _json

_adapter_save_path = str(CFG.phase1_adapter_dir)
CFG.phase1_adapter_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapter to: {_adapter_save_path}")
model.save_pretrained(_adapter_save_path, safe_serialization=True)

# Save processor alongside adapter for self-contained reloading
print(f"Saving processor to   : {_adapter_save_path}")
processor.save_pretrained(_adapter_save_path)

# Embed PHASE1_TEMPLATE_HASH in adapter_config so Phase 3 can verify
# the prompt template at inference time matches the one used during training.
_adapter_cfg_path = CFG.phase1_adapter_dir / "adapter_config.json"
if _adapter_cfg_path.exists():
    with open(_adapter_cfg_path, "r") as _f:
        _adapter_cfg = _json.load(_f)
    _adapter_cfg["phase1_template_hash"] = PHASE1_TEMPLATE_HASH
    with open(_adapter_cfg_path, "w") as _f:
        _json.dump(_adapter_cfg, _f, indent=2)
    print(f"  Wrote PHASE1_TEMPLATE_HASH={PHASE1_TEMPLATE_HASH!r} to adapter_config.json")
else:
    log.warning(
        "adapter_config.json not found after save_pretrained. "
        "Template hash NOT embedded. Phase 3 drift check will be skipped."
    )

# Verify files exist
_expected_files = ["adapter_model.safetensors", "adapter_config.json"]
_missing = [f for f in _expected_files if not (CFG.phase1_adapter_dir / f).exists()]
if _missing:
    raise RuntimeError(
        f"Missing adapter files after save: {_missing}. "
        f"Check {_adapter_save_path} for partial writes."
    )

print(f"\nok  LoRA adapter saved successfully.")
print(f"    Files: {[f.name for f in CFG.phase1_adapter_dir.iterdir()]}")
print(f"    Template hash: {PHASE1_TEMPLATE_HASH}")


In [ ]:
# -- S10.3  Post-training generation test (qualitative) -----------------------
# Verifies the fine-tuned model produces valid JSON edit predictions.
# This is a qualitative smoke test -- NOT a quantitative evaluation (Phase 4).
#
# We run 3 samples from the dataset through greedy decoding and verify:
#   1. Output is valid JSON
#   2. JSON contains required keys: edit_type, bbox, instruction
#   3. bbox is a list of 4 numbers in [0, 1000] range
#   4. edit_type matches the ground-truth edit_type
#
# Model is set to eval() mode and back to train() after the test.
# Uses the same build_messages() path as training -- no template drift.

import json as _json_mod

model.eval()

_test_samples = _full_raw[:3]   # use first 3 from training set (easy sanity)
_n_pass = 0
_n_fail = 0

print("--- Post-training generation test ---")
print("Running greedy decoding on 3 samples...\n")

for _i, _samp in enumerate(_test_samples):
    _src_path = _samp["source_image_path"]
    try:
        _image   = Image.open(_src_path).convert("RGB")
    except Exception as _e:
        print(f"[{_i}] SKIP -- cannot open image: {_e}")
        continue

    _messages = build_messages(
        _image,
        _samp["instruction"],
        annotations = _samp.get("annotations") or [],
        seg_dims    = _samp.get("seg_dims"),
    )

    _text_prompt = processor.apply_chat_template(
        _messages, tokenize=False, add_generation_prompt=True
    )
    _image_inputs, _video_inputs = process_vision_info(_messages)

    _inputs = processor(
        text           = [_text_prompt],
        images         = _image_inputs if _image_inputs else None,
        videos         = _video_inputs if _video_inputs else None,
        padding        = True,
        return_tensors = "pt",
    )

    _dev = str(next(model.parameters()).device)
    _inputs = {k: v.to(_dev) if isinstance(v, torch.Tensor) else v for k, v in _inputs.items()}

    with torch.no_grad():
        _gen_ids = model.generate(
            **_inputs,
            max_new_tokens = 128,
            do_sample      = False,    # greedy
            temperature    = 1.0,
            pad_token_id   = processor.tokenizer.eos_token_id,
        )

    # Decode only the new tokens (strip input prompt)
    _prompt_len = _inputs["input_ids"].shape[1]
    _new_tokens = _gen_ids[0][_prompt_len:]
    _raw_output = processor.tokenizer.decode(_new_tokens, skip_special_tokens=True).strip()

    # Validate JSON
    _json_ok   = False
    _keys_ok   = False
    _bbox_ok   = False
    _type_ok   = False
    _parsed    = None

    try:
        _parsed  = _json_mod.loads(_raw_output)
        _json_ok = True
        _req_keys = {"edit_type", "bbox", "instruction"}
        _keys_ok  = _req_keys.issubset(_parsed.keys())
        if _keys_ok:
            _bbox = _parsed["bbox"]
            _bbox_ok = (
                isinstance(_bbox, list)
                and len(_bbox) == 4
                and all(isinstance(v, (int, float)) and 0 <= v <= 1000 for v in _bbox)
            )
            _type_ok = _parsed.get("edit_type") == _samp["edit_type"]
    except Exception:
        pass

    _status = "PASS" if (_json_ok and _keys_ok and _bbox_ok) else "FAIL"
    if _status == "PASS":
        _n_pass += 1
    else:
        _n_fail += 1

    print(f"[{_i}] {_status}")
    print(f"     gt edit_type : {_samp['edit_type']!r}")
    print(f"     gt bbox      : {_samp['bbox']}")
    print(f"     raw output   : {_raw_output[:200]!r}")
    if _parsed:
        print(f"     pred edit_type: {_parsed.get('edit_type')!r}  match={_type_ok}")
        print(f"     pred bbox      : {_parsed.get('bbox')}  valid={_bbox_ok}")
    print()

print(f"Results: {_n_pass}/3 passed JSON validation")
if _n_pass == 0:
    log.warning(
        "No samples passed JSON validation. The model may need more epochs, "
        "or the response_start detection may be incorrect. "
        "Check label masking in tokenize_sample -- run cell_18 again."
    )
elif _n_pass < 3:
    log.warning(
        f"Only {_n_pass}/3 samples produced valid JSON. "
        "Some overfitting or template inconsistency may be present. "
        "Proceeding -- this is a qualitative check, not a hard gate."
    )
else:
    print("ok  All 3 samples produced valid JSON with required keys.")

model.train()


In [ ]:
_full_raw = load_vlm_dataset(CFG.data_dir / "vlm_dataset_phase1v2.json")

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, str(CFG.phase1_adapter_dir), is_trainable=False)
model.eval()

# Section 11 -- VLM Hidden-State Extraction and Caching

This section extracts and caches the Qwen2.5-VL hidden states that will be used as
conditioning vectors in Phase 2 (Stable Diffusion inpainting bridge training).

**Why pre-cache hidden states (not compute on-the-fly in Phase 2)?**
- SD training runs thousands of steps; recomputing VLM hidden states in the inner loop
  would add ~200ms per step and require keeping the VLM on GPU simultaneously with the UNet.
- Pre-caching allows Phase 2 to load compact (2048,) float32 vectors from Drive, using
  negligible memory and storage (~80 MB for 10k samples).

**What is extracted:**
- From a **user-turn-only** forward pass (no assistant response) with `output_hidden_states=True`
- Last hidden layer: `last_hidden_state` tensor of shape `(1, seq_len, hidden_dim)`
- **Token filtering**: Tokens with ID >= `QWEN_SPECIAL_START` (151643) are excluded.
  This removes image tokens (`<|image_pad|>`) and chat-template special tokens
  (`<|im_start|>`, `<|im_end|>`, etc.), retaining only the natural-language instruction tokens.
- **Mean pooling** over the filtered text tokens -> `(hidden_dim,)` = `(2048,)` float32 vector

**Shard format (restart-safe):**
- Shards saved as `shard_{N:04d}.pt` under `CFG.hidden_states_dir`
- Each shard: `{"hidden_states": Tensor(shard_size, 2048), "sample_ids": [str, ...]}`
- Manifest updated with `shard_id` and `row_index` for each sample
- Samples already in the manifest with `shard_id` set are skipped on re-run

**Train/inference consistency:**
- Extraction uses exactly `build_messages(image, instruction)` (user turn only) --
  the same function called at Phase 3 inference time.
- `PHASE1_TEMPLATE_HASH` is saved alongside each shard set so Phase 3 can detect drift.


In [ ]:
# -- S11.1  Token filtering constants + extract_hidden_state_for_sample() -----
#
# SPEC REQUIREMENT: "Qwen hidden-state extraction must exclude image tokens and
# special/template tokens before mean pooling."
#
# QWEN_SPECIAL_START = 151643 (defined in S3.1 / cell_09.py):
#   - IDs < 151643 : regular vocabulary (text tokens) -- KEEP for mean pooling
#   - IDs >= 151643: special tokens (<|image_pad|>, <|im_start|>, etc.) -- EXCLUDE
#
# Why user-turn-only (no assistant response):
#   At Phase 3 inference time we do NOT have the assistant response.
#   Training the Phase 2 bridge on hidden states that include the response would
#   create a train/inference mismatch -- the bridge would learn to condition on
#   information unavailable at runtime.
#   SPEC says: "user-turn forward pass" for conditioning extraction.
#
# Hidden state source: model's LAST hidden layer (index -1 in hidden_states tuple).
#   Layer -1 is the full-context representation AFTER all transformer blocks.
#   This is the standard choice for downstream conditioning; earlier layers
#   are more syntactic and less semantically rich.
#
# Output: float32 (2048,) vector, moved to CPU before return to avoid GPU
# memory accumulation across the full dataset extraction loop.

import torch
from typing import Optional

QWEN_SPECIAL_START = 151643
# QWEN_SPECIAL_START already defined in cell_09.py (S3.1), asserted here for safety
assert QWEN_SPECIAL_START == 151643, (
    f"QWEN_SPECIAL_START={QWEN_SPECIAL_START} -- expected 151643. "
    "Re-run cell_09.py (S3.1) if this constant was changed."
)


def extract_hidden_state_for_sample(
    sample_meta: dict,
    model,
    processor,
    device: str,
) -> Optional[torch.Tensor]:
    '''Extract a (hidden_dim,) float32 conditioning vector for one training sample.

    Runs a user-turn-only forward pass through the VLM with output_hidden_states=True,
    then mean-pools the text-only token representations from the last hidden layer.

    This is the IDENTICAL path used during Phase 3 inference -- see build_messages().
    Do NOT modify this function without updating the Phase 3 counterpart.

    Args:
        sample_meta: dict with source_image_path and instruction (from build_vlm_dataset)
        model:       Fine-tuned Qwen2.5-VL on device (eval mode recommended)
        processor:   AutoProcessor matching the model
        device:      "cuda" or "cpu"

    Returns:
        Tensor of shape (hidden_dim,) dtype=float32 on CPU, or None on failure.
        Returns None (not raises) so the caller can skip bad samples without
        crashing the full extraction loop.
    '''
    try:
        # Load image
        src_path = sample_meta["source_image_path"]
        image    = Image.open(src_path).convert("RGB")

        # Build user-turn messages (SAME function as Phase 3 inference).
        # v1.3: pass annotations + seg_dims so hidden states are extracted
        # from the same full-scene prompt the model was trained on.
        # Do NOT add assistant response -- creates train/inference mismatch.
        messages = build_messages(
            image,
            sample_meta["instruction"],
            annotations = sample_meta.get("annotations") or [],
            seg_dims    = sample_meta.get("seg_dims"),
        )

        # Format with add_generation_prompt=True to get the prompt-only text
        # (no assistant response tokens appended)
        text_prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text           = [text_prompt],
            images         = image_inputs if image_inputs else None,
            videos         = video_inputs if video_inputs else None,
            padding        = False,
            return_tensors = "pt",
        )

        input_ids = inputs["input_ids"].to(device)
        attn_mask = inputs["attention_mask"].to(device)

        # Move image patches to device if present
        extra = {}
        if "pixel_values" in inputs:
            extra["pixel_values"]   = inputs["pixel_values"].to(device)
        if "image_grid_thw" in inputs:
            extra["image_grid_thw"] = inputs["image_grid_thw"].to(device)

        # Forward pass -- output_hidden_states=True returns tuple of per-layer tensors
        with torch.no_grad():
            outputs = model(
                input_ids         = input_ids,
                attention_mask    = attn_mask,
                output_hidden_states = True,
                **extra,
            )

        # Last hidden layer: shape (1, seq_len, hidden_dim)
        last_hidden = outputs.hidden_states[-1]  # tuple index -1 = last layer
        assert last_hidden.ndim == 3 and last_hidden.shape[0] == 1, (
            f"Unexpected last_hidden shape: {last_hidden.shape}"
        )

        seq_len    = last_hidden.shape[1]
        id_list    = input_ids[0].tolist()   # flat list of token IDs, length seq_len

        # Build boolean mask: True for text tokens, False for special/image tokens
        text_mask = torch.tensor(
            [tok_id < QWEN_SPECIAL_START for tok_id in id_list],
            dtype  = torch.bool,
            device = device,
        )   # shape (seq_len,)

        n_text_tokens = text_mask.sum().item()
        if n_text_tokens == 0:
            log.warning(
                f"No text tokens found for {sample_meta.get('sample_id','?')} "
                f"after filtering (all {seq_len} tokens have ID >= {QWEN_SPECIAL_START}). "
                "Skipping sample."
            )
            return None

        # Mean pool text tokens: (1, seq_len, hidden_dim) -> (hidden_dim,)
        # Expand mask to broadcast over hidden_dim
        text_hidden   = last_hidden[0][text_mask]  # (n_text, hidden_dim)
        pooled        = text_hidden.mean(dim=0)     # (hidden_dim,)

        assert pooled.shape == (model.config.hidden_size,), (
            f"Pooled hidden state shape {pooled.shape} != "
            f"expected ({model.config.hidden_size},)"
        )

        # Return float32 on CPU (GPU memory must not accumulate across all samples)
        return pooled.float().cpu()

    except Exception as e:
        log.warning(
            f"extract_hidden_state_for_sample failed for "
            f"{sample_meta.get('sample_id','?')}: {e}"
        )
        return None


print("extract_hidden_state_for_sample() defined.")
print(f"  QWEN_SPECIAL_START = {QWEN_SPECIAL_START}  (text tokens: ID < threshold)")
print(f"  Output shape       : ({CFG.vlm_hidden_dim},)  float32  on CPU")
print(f"  Forward pass       : user-turn-only (no assistant response)")
print(f"  Pooling            : mean over text tokens, last hidden layer")


# -- Quick single-sample test -------------------------------------------------
print("\nRunning single-sample extraction test...")
model.eval()
_test_hs = extract_hidden_state_for_sample(_full_raw[0], model, processor, "cuda")
assert _test_hs is not None, "Extraction returned None on first sample -- check logs"
assert _test_hs.shape == (CFG.vlm_hidden_dim,), (
    f"Expected shape ({CFG.vlm_hidden_dim},), got {_test_hs.shape}"
)
assert not _test_hs.isnan().any(), "NaN in extracted hidden state"
assert _test_hs.dtype == torch.float32, f"Expected float32, got {_test_hs.dtype}"
print(f"ok  Extraction test passed.")
print(f"    Shape : {tuple(_test_hs.shape)}")
print(f"    Dtype : {_test_hs.dtype}")
print(f"    Range : [{_test_hs.min().item():.4f}, {_test_hs.max().item():.4f}]")
model.train()


In [ ]:
# -- S11.2  cache_vlm_hidden_states() -- sharded Drive caching ----------------
#
# SPEC REQUIREMENT: "All expensive outputs must be cached to Drive."
# "Phase 2 must use pre-cached VLM hidden states by default."
#
# Restart safety design:
#   - We read the manifest before starting. Any sample with shard_id != None
#     already has a cached hidden state and is skipped.
#   - We accumulate a shard buffer in memory. When it reaches CFG.hs_shard_size
#     samples, we flush it to Drive as shard_{N:04d}.pt and update the manifest.
#   - If Colab dies mid-shard, the partial buffer is lost but all flushed shards
#     and manifest entries are intact. Re-running skips already-cached samples
#     and only extracts the remainder.
#   - A metadata file (hidden_states_meta.json) stores the template hash and
#     total shard count for downstream validation by Phase 2/3.
#
# Shard file format:
#   {
#     "hidden_states": Tensor(N, hidden_dim),  # float32
#     "sample_ids":    [str, str, ...]          # length N
#   }
#
# Manifest update: for each cached sample, manifest[i]["shard_id"] = "shard_0003"
#                                          manifest[i]["row_index"] = 42
# These two fields let Phase 2 load exactly the right row for each sample.

import torch
from pathlib import Path
from typing import List


def cache_vlm_hidden_states(
    raw_samples:     list,
    manifest:        list,
    model,
    processor,
    hidden_states_dir: Path,
    manifest_path:   Path,
    shard_size:      int = None,
    device:          str = "cuda",
) -> dict:
    '''Extract and cache VLM hidden states for all samples, with restart safety.

    Args:
        raw_samples:       List of sample dicts from build_vlm_dataset().
                           Must align 1-to-1 with manifest (same order, same sample_id).
        manifest:          Full manifest list (loaded from filtered_manifest_path).
                           Updated IN-PLACE with shard_id and row_index.
        model:             Fine-tuned Qwen2.5-VL (eval mode set internally).
        processor:         AutoProcessor.
        hidden_states_dir: Drive directory for shard .pt files.
        manifest_path:     Path to write updated manifest JSON.
        shard_size:        Samples per shard file (default CFG.hs_shard_size).
        device:            "cuda" or "cpu".

    Returns:
        Summary dict: {total, cached_before, extracted, skipped, shards_written}.

    Raises:
        RuntimeError if manifest and raw_samples have different sample_id ordering.
    '''
    if shard_size is None:
        shard_size = CFG.hs_shard_size

    hidden_states_dir.mkdir(parents=True, exist_ok=True)

    # -- Build sample_id -> manifest_index lookup ----------------------------
    # Verify alignment between raw_samples and manifest
    manifest_by_id = {m["sample_id"]: (i, m) for i, m in enumerate(manifest)}

    for rs in raw_samples:
        sid = rs["sample_id"]
        if sid not in manifest_by_id:
            raise RuntimeError(
                f"sample_id {sid!r} in raw_samples not found in manifest. "
                "raw_samples and manifest must be built from the same filtered manifest."
            )

    # -- Count already-cached samples ----------------------------------------
    n_already_cached = sum(
        1 for m in manifest if m.get("shard_id") is not None
    )
    print(f"Total samples       : {len(raw_samples)}")
    print(f"Already cached      : {n_already_cached}")
    print(f"To extract          : {len(raw_samples) - n_already_cached}")
    print(f"Shard size          : {shard_size}")
    print(f"Output dir          : {hidden_states_dir}")

    model.eval()

    # -- Count existing shards to set next shard index -----------------------
    _existing_shards = sorted(hidden_states_dir.glob("shard_*.pt"))
    next_shard_idx   = len(_existing_shards)

    # -- Accumulation buffer -------------------------------------------------
    buf_hidden_states: List[torch.Tensor] = []
    buf_sample_ids:    List[str]          = []

    stats = {
        "total":          len(raw_samples),
        "cached_before":  n_already_cached,
        "extracted":      0,
        "skipped":        0,
        "shards_written": 0,
    }

    def _flush_shard():
        '''Write current buffer to a new shard file and update manifest.'''
        if not buf_hidden_states:
            return

        shard_name = f"shard_{next_shard_idx:04d}"
        shard_path = hidden_states_dir / f"{shard_name}.pt"

        shard_tensor = torch.stack(buf_hidden_states, dim=0)  # (N, hidden_dim)
        assert shard_tensor.ndim == 2 and shard_tensor.shape[1] == CFG.vlm_hidden_dim, (
            f"Shard tensor shape {shard_tensor.shape} unexpected. "
            f"Expected (N, {CFG.vlm_hidden_dim})"
        )

        torch.save(
            {"hidden_states": shard_tensor, "sample_ids": list(buf_sample_ids)},
            shard_path,
        )

        # Update manifest entries for all samples in this shard
        for row_idx, sid in enumerate(buf_sample_ids):
            mf_idx, _ = manifest_by_id[sid]
            manifest[mf_idx]["shard_id"]   = shard_name
            manifest[mf_idx]["row_index"]  = row_idx

        # Persist updated manifest atomically after each shard flush
        save_json(manifest, manifest_path)

        stats["shards_written"] += 1
        print(f"  Flushed {shard_name}.pt  ({len(buf_sample_ids)} samples)")

        buf_hidden_states.clear()
        buf_sample_ids.clear()

    # -- Main extraction loop ------------------------------------------------
    for i, rs in enumerate(raw_samples):
        sid = rs["sample_id"]
        mf_idx, mf_entry = manifest_by_id[sid]

        # Skip already-cached
        if mf_entry.get("shard_id") is not None:
            continue

        hs = extract_hidden_state_for_sample(rs, model, processor, device)

        if hs is None:
            log.warning(f"Skipping {sid} -- extraction returned None.")
            stats["skipped"] += 1
            continue

        buf_hidden_states.append(hs)
        buf_sample_ids.append(sid)
        stats["extracted"] += 1

        # Flush when buffer is full
        if len(buf_hidden_states) >= shard_size:
            _flush_shard()
            next_shard_idx += 1

        if (i + 1) % 100 == 0:
            pct = (stats["extracted"] + stats["cached_before"]) / len(raw_samples) * 100
            print(f"  [{i+1}/{len(raw_samples)}]  extracted={stats['extracted']}  "
                  f"skipped={stats['skipped']}  ({pct:.1f}% total done)")

    # Flush remaining buffer (partial shard)
    if buf_hidden_states:
        _flush_shard()

    # -- Save metadata for Phase 2/3 validation ------------------------------
    meta_path = hidden_states_dir / "hidden_states_meta.json"
    meta = {
        "phase1_template_hash": PHASE1_TEMPLATE_HASH,
        "vlm_model_id":         CFG.vlm_model_id,
        "hidden_dim":           CFG.vlm_hidden_dim,
        "total_samples":        stats["extracted"] + n_already_cached,
        "n_shards":             next_shard_idx + (1 if buf_hidden_states else 0),
        "shard_size":           shard_size,
        "qwen_special_start":   QWEN_SPECIAL_START,
        "extraction_note": (
            "Token IDs >= QWEN_SPECIAL_START excluded before mean pooling. "
            "User-turn-only forward pass (no assistant response tokens)."
        ),
    }
    save_json(meta, meta_path)
    print(f"\n  Saved metadata to: {meta_path}")

    model.train()
    return stats


print("cache_vlm_hidden_states() defined.")
print(f"  Shard size    : {CFG.phase2_shard_size}")
print(f"  Output dir    : {CFG.hidden_states_dir}")
print(f"  Restart safe  : yes (skips samples with existing shard_id in manifest)")


In [ ]:
# -- S11.3  Run hidden-state extraction and verify shards ---------------------
# This cell runs the full extraction pipeline on all training samples.
# Expected runtime: ~15-30 seconds per sample on T4 (image load + VLM forward).
# For 10k samples: ~40-80 hours on T4. Use A100/V100 for faster extraction,
# OR extract in batches across multiple sessions (restart safety handles this).
#
# IMPORTANT: If you killed Colab mid-extraction, just re-run this cell.
#   cache_vlm_hidden_states() reads the manifest, skips already-cached samples,
#   and continues from where it left off.
#
# After extraction, this cell verifies:
#   1. All shards load correctly with correct shapes
#   2. All manifest entries have shard_id + row_index set
#   3. A random sample round-trips (manifest -> shard -> hidden_state)
#   4. metadata JSON exists with correct template hash

import random as _random

print("="*60)
print("S11.3  Running VLM hidden-state extraction")
print("="*60)

# -- Load full manifest (the source of truth for shard_id tracking) ----------
_manifest = load_json(CFG.filtered_manifest_path)
print(f"Manifest loaded: {len(_manifest)} samples")

# Verify _full_raw was built earlier (cell_22.py builds it)
if "_full_raw" not in dir() or _full_raw is None:
    print("Rebuilding _full_raw from manifest (cell_22.py may not have run)...")
    _full_raw = build_vlm_dataset(
        manifest_path  = CFG.filtered_manifest_path,
        data_dir       = CFG.data_dir,
        sample_limit   = None,
        warn_on_fallback = False,
    )

# -- Run extraction ----------------------------------------------------------
_stats = cache_vlm_hidden_states(
    raw_samples        = _full_raw,
    manifest           = _manifest,
    model              = model,
    processor          = processor,
    hidden_states_dir  = CFG.hidden_states_dir,
    manifest_path      = CFG.filtered_manifest_path,
    shard_size         = CFG.phase2_shard_size,
    device             = "cuda",
)

print("\n--- Extraction summary ---")
for k, v in _stats.items():
    print(f"  {k:<20s}: {v}")

# -- Verify shards -----------------------------------------------------------
print("\nVerifying shard files...")
_shard_files = sorted(CFG.hidden_states_dir.glob("shard_*.pt"))
assert len(_shard_files) > 0, (
    f"No shard files found in {CFG.hidden_states_dir}. Extraction may have failed."
)

_total_rows = 0
for _sf in _shard_files:
    _data = torch.load(_sf, map_location="cpu")
    assert "hidden_states" in _data and "sample_ids" in _data, (
        f"Shard {_sf.name} missing expected keys: {list(_data.keys())}"
    )
    _hs = _data["hidden_states"]
    assert _hs.ndim == 2 and _hs.shape[1] == CFG.vlm_hidden_dim, (
        f"Shard {_sf.name}: unexpected shape {_hs.shape}"
    )
    assert len(_data["sample_ids"]) == _hs.shape[0], (
        f"Shard {_sf.name}: sample_ids length {len(_data['sample_ids'])} "
        f"!= hidden_states rows {_hs.shape[0]}"
    )
    assert not _hs.isnan().any(), f"NaN values in shard {_sf.name}"
    _total_rows += _hs.shape[0]
    print(f"  ok  {_sf.name}: shape={tuple(_hs.shape)}, dtype={_hs.dtype}")

print(f"\n  Total rows across all shards: {_total_rows}")

# -- Verify manifest coverage ------------------------------------------------
print("\nVerifying manifest coverage...")
_n_with_shard    = sum(1 for m in _manifest if m.get("shard_id") is not None)
_n_without_shard = sum(1 for m in _manifest if m.get("shard_id") is None)
print(f"  Manifest entries with shard_id   : {_n_with_shard}")
print(f"  Manifest entries without shard_id: {_n_without_shard}")
if _n_without_shard > 0:
    log.warning(
        f"{_n_without_shard} manifest entries still missing shard_id. "
        "These samples had extraction failures (see logs above). "
        "Phase 2 will skip these samples -- verify this is acceptable."
    )

# -- Round-trip test: pick a random cached sample ----------------------------
print("\nRound-trip test: manifest -> shard -> hidden_state...")
_cached_entries = [m for m in _manifest if m.get("shard_id") is not None]
assert len(_cached_entries) > 0, "No cached entries in manifest to test"

_rt_entry  = _random.choice(_cached_entries)
_shard_path = CFG.hidden_states_dir / f"{_rt_entry['shard_id']}.pt"
assert _shard_path.exists(), (
    f"Shard file {_shard_path} referenced in manifest but not found on disk."
)

_rt_data = torch.load(_shard_path, map_location="cpu")
_row_idx = _rt_entry["row_index"]
assert _row_idx < len(_rt_data["sample_ids"]), (
    f"row_index={_row_idx} out of range for shard with {len(_rt_data['sample_ids'])} rows"
)
assert _rt_data["sample_ids"][_row_idx] == _rt_entry["sample_id"], (
    f"Round-trip mismatch: expected sample_id={_rt_entry['sample_id']!r}, "
    f"got {_rt_data['sample_ids'][_row_idx]!r}"
)
_rt_hs = _rt_data["hidden_states"][_row_idx]
assert _rt_hs.shape == (CFG.vlm_hidden_dim,), (
    f"Round-trip hidden state shape {_rt_hs.shape} != ({CFG.vlm_hidden_dim},)"
)
print(f"  ok  sample_id={_rt_entry['sample_id']!r}")
print(f"      shard={_rt_entry['shard_id']}, row={_row_idx}")
print(f"      hidden_state shape={tuple(_rt_hs.shape)}, dtype={_rt_hs.dtype}")

# -- Verify metadata file ----------------------------------------------------
_meta_path = CFG.hidden_states_dir / "hidden_states_meta.json"
assert _meta_path.exists(), (
    f"hidden_states_meta.json not found in {CFG.hidden_states_dir}"
)
_meta = load_json(_meta_path)
assert _meta["phase1_template_hash"] == PHASE1_TEMPLATE_HASH, (
    f"Template hash mismatch in metadata!\n"
    f"  Stored : {_meta['phase1_template_hash']}\n"
    f"  Current: {PHASE1_TEMPLATE_HASH}\n"
    "This means the prompt template changed after caching. "
    "Re-extract all hidden states or revert the template."
)
print(f"\n  ok  Metadata hash matches: {PHASE1_TEMPLATE_HASH}")

print(f"\nok  S11 complete. {_total_rows} hidden states cached across {len(_shard_files)} shards.")
print(f"    Manifest updated at: {CFG.filtered_manifest_path}")
print(f"    Ready for Phase 2 bridge training.")


# Section 11.4 -- Re-extraction after resumed Phase 1 training (v2 cache)

Use this section *only* when you have already extracted a v1 hidden-state cache
and want to re-extract with a more-trained adapter.  It writes a parallel
`vlm_hidden_states_v2/` cache and a `samples_filtered_v2.json` manifest, leaving
the original v1 cache untouched.

**Why a separate cache?**  `cache_vlm_hidden_states()` skips any sample whose
manifest entry already has `shard_id` set -- restart-safety for an interrupted
extraction.  After v1 is complete, every entry has `shard_id`, so re-running
extraction against the v1 manifest would silently skip 100% of samples.  The
embedded `phase1_template_hash` only catches prompt-template drift, not changes
in adapter weights, so we cannot rely on it to detect a stale cache.

**Workflow:**
1. In a fresh Colab session, re-run S0--S9 (config, install, model load).
2. Load the resumed adapter:
   `model = PeftModel.from_pretrained(base_model, str(CFG.phase1_adapter_dir), is_trainable=False)`
3. Run **S11.4a** to create the v2 manifest.
4. Run **S11.4b** to extract.  Restart-safe -- re-running resumes from where it left off.
5. Point Phase 2 at `CFG.filtered_manifest_v2_path` and `CFG.hidden_states_dir_v2`.


In [ ]:
# -- Override CFG.phase1_adapter_dir to match where the adapter actually lives ---
# The default in S0.1 points to ckpt_phase1/lora_adapter, but in this run the
# adapter was saved directly under ckpt_phase1, so we redirect the property.
# Property reassignment goes on the CLASS (descriptor protocol), not the instance.
from pathlib import Path

PipelineConfig.phase1_adapter_dir = property(
    lambda self: Path("/content/drive/MyDrive/img_edit_pipeline/checkpoints/phase1_vlm")
)

# Verify
print("phase1_adapter_dir ->", CFG.phase1_adapter_dir)
assert (CFG.phase1_adapter_dir / "adapter_config.json").exists(), (
    f"adapter_config.json not found at {CFG.phase1_adapter_dir}"
)
print("ok  adapter_config.json found.")

In [ ]:
# -- S11.4.0  Locate the Phase 1 LoRA adapter on Drive -----------------------
# The default CFG.phase1_adapter_dir is data/checkpoints/phase1_vlm/lora_adapter.
# If your previous training session saved the adapter to a different path
# (the older notebook had no default and required a manual override), we need
# to find it before S11.4a runs its template-hash check.
#
# This cell:
#   1. Scans CFG.drive_root recursively for adapter_config.json files.
#   2. Prints every match with size and modification time so you can spot
#      the right one.
#   3. If the default CFG.phase1_adapter_dir already contains adapter_config.json,
#      we're done -- nothing to override.
#   4. Otherwise it prints the exact one-line override you need to run.
#
# Re-runnable; never modifies state.

from pathlib import Path as _Path
import datetime as _dt

print("=" * 60)
print("S11.4.0  Locating the Phase 1 LoRA adapter on Drive")
print("=" * 60)
print(f"Default CFG.phase1_adapter_dir : {CFG.phase1_adapter_dir}")
print()

# 1. Direct check at the default location
_default_cfg = CFG.phase1_adapter_dir / "adapter_config.json"
if _default_cfg.exists():
    print(f"ok  Adapter found at the default path -- no override needed.")
    print(f"    {_default_cfg}")
else:
    print(f"WARN  No adapter_config.json at default path.")
    print(f"      Scanning {CFG.drive_root} for candidates...\n")

    # 2. Recursive scan for any adapter_config.json under the Drive root.
    #    We look for the file (not the dir) because PEFT always writes it.
    _hits = sorted(CFG.drive_root.rglob("adapter_config.json"))
    if not _hits:
        raise RuntimeError(
            f"No adapter_config.json files found anywhere under {CFG.drive_root}. "
            "Either the adapter was saved to a path outside drive_root, or S10.2 "
            "did not actually run successfully in your training session. "
            "Check Drive manually and confirm before proceeding."
        )

    print(f"Found {len(_hits)} adapter_config.json file(s):\n")
    for _h in _hits:
        _stat = _h.stat()
        _mtime = _dt.datetime.fromtimestamp(_stat.st_mtime).strftime("%Y-%m-%d %H:%M")
        _kb    = _stat.st_size / 1024
        print(f"  {_h}")
        print(f"    size = {_kb:.1f} KB   modified = {_mtime}")
        # Peek at the template hash so the user can confirm provenance
        try:
            import json as _json
            with open(_h, "r") as _f:
                _cfg = _json.load(_f)
            _ph = _cfg.get("phase1_template_hash", "<MISSING>")
            print(f"    phase1_template_hash = {_ph!r}")
        except Exception as _e:
            print(f"    (could not read: {_e})")
        print()

    # 3. Suggest the override -- pick the most recently modified candidate
    _newest = max(_hits, key=lambda p: p.stat().st_mtime)
    _adapter_dir = _newest.parent
    print("To use the most recently modified adapter, paste this into a NEW")
    print("cell before running S11.4a, then re-run S11.4a:\n")
    print(f"    from pathlib import Path")
    print(f'    PipelineConfig.phase1_adapter_dir = property(')
    print(f'        lambda self: Path({str(_adapter_dir)!r})')
    print(f'    )')
    print(f"    print('phase1_adapter_dir overridden ->', CFG.phase1_adapter_dir)\n")
    print("Or, if you prefer to keep CFG pristine, just MOVE the adapter files")
    print(f"on Drive into the default directory:")
    print(f"    {CFG.phase1_adapter_dir}\n")

    raise RuntimeError(
        "Adapter not at default path. See the override snippet printed above, "
        "apply it in a new cell, then re-run S11.4a."
    )


In [ ]:
# -- S11.4a  Prepare v2 hidden-states cache after resumed training -----------
# Why this cell exists:
#   cache_vlm_hidden_states() skips any sample where manifest[i]["shard_id"]
#   is not None.  After our first (v1) extraction every entry has shard_id
#   set, so re-running with a re-trained adapter would extract zero samples.
#   This cell creates a parallel manifest with shard_id reset to None so the
#   re-extraction actually runs, while leaving v1 manifest + shards untouched.
#
# Fail-fast checks (do not relax these -- they exist to prevent silent corruption):
#   - v1 cache must already be complete (otherwise you should be running S11.3).
#   - v2 directory must not already contain shards (would mix old + new weights).
#   - Adapter on disk must match the loaded model's template hash.
#
# Idempotency:
#   Re-running this cell after the v2 manifest is created is a no-op aside
#   from re-printing summary stats; it never overwrites an existing manifest.

import json as _json
from pathlib import Path as _Path

print("=" * 60)
print("S11.4a  Preparing v2 hidden-states cache")
print("=" * 60)

# -- 1. Verify v1 cache exists and is non-empty ------------------------------
_v1_manifest = load_json(CFG.filtered_manifest_path)
_v1_total    = len(_v1_manifest)
_v1_cached   = sum(1 for m in _v1_manifest if m.get("shard_id") is not None)
print(f"v1 manifest         : {CFG.filtered_manifest_path}")
print(f"  total samples     : {_v1_total}")
print(f"  with shard_id     : {_v1_cached}")
assert _v1_cached > 0, (
    "v1 cache appears empty (no manifest entries have shard_id). "
    "If you have not run a first extraction yet, run S11.3 instead -- "
    "this cell is only for the post-resumed-training re-extraction path."
)

# -- 2. Verify the v2 directory does not already contain shards --------------
CFG.hidden_states_dir_v2.mkdir(parents=True, exist_ok=True)
_existing_v2_shards = sorted(CFG.hidden_states_dir_v2.glob("shard_*.pt"))
if _existing_v2_shards and not CFG.filtered_manifest_v2_path.exists():
    raise RuntimeError(
        f"Found {len(_existing_v2_shards)} shard files in {CFG.hidden_states_dir_v2} "
        "but no v2 manifest. This indicates a corrupted/orphaned v2 state. "
        "Inspect manually and either delete the shards to start over, or "
        "restore the v2 manifest from a Drive backup before re-running."
    )

# -- 3. Verify the adapter on disk matches the template hash in this notebook
_adapter_cfg_path = CFG.phase1_adapter_dir / "adapter_config.json"
assert _adapter_cfg_path.exists(), (
    f"adapter_config.json not found at {_adapter_cfg_path}. "
    "Did S10.2 run successfully in the training session?"
)
with open(_adapter_cfg_path, "r") as _f:
    _saved_hash = _json.load(_f).get("phase1_template_hash")
assert _saved_hash == PHASE1_TEMPLATE_HASH, (
    f"Template hash drift between training and extraction:\n"
    f"  saved in adapter_config.json : {_saved_hash!r}\n"
    f"  current PHASE1_TEMPLATE_HASH : {PHASE1_TEMPLATE_HASH!r}\n"
    "Hidden states extracted with a different prompt template than the model "
    "was trained on would create a Phase 2/3 train-inference mismatch. Refuse."
)
print(f"template hash check : ok  ({PHASE1_TEMPLATE_HASH!r})")

# -- 4. Create or reuse the v2 manifest --------------------------------------
if CFG.filtered_manifest_v2_path.exists():
    _v2_manifest = load_json(CFG.filtered_manifest_v2_path)
    assert len(_v2_manifest) == _v1_total, (
        f"existing v2 manifest length {len(_v2_manifest)} != v1 length {_v1_total}. "
        "Delete the v2 manifest and re-run this cell to rebuild it from v1."
    )
    _v2_cached = sum(1 for m in _v2_manifest if m.get("shard_id") is not None)
    print(f"v2 manifest exists  : {CFG.filtered_manifest_v2_path}")
    print(f"  already cached    : {_v2_cached}/{_v1_total}")
    print("  -> S11.4b will resume extraction from the next uncached entry.")
else:
    print(f"creating v2 manifest: {CFG.filtered_manifest_v2_path}")
    # Deep copy via JSON round-trip so we cannot accidentally mutate v1 in memory.
    _v2_manifest = _json.loads(_json.dumps(_v1_manifest))
    for m in _v2_manifest:
        m["shard_id"]  = None
        m["row_index"] = None
    save_json(_v2_manifest, CFG.filtered_manifest_v2_path)
    # Sanity check after write
    _v2_check = load_json(CFG.filtered_manifest_v2_path)
    assert all(m.get("shard_id") is None for m in _v2_check), (
        "v2 manifest still has shard_id set after reset -- write failed?"
    )
    print(f"  reset shard_id/row_index on {len(_v2_check)} entries.")

# -- 5. Record provenance metadata for Phase 2/3 audits ----------------------
_v2_meta = {
    "source_adapter_dir":   str(CFG.phase1_adapter_dir),
    "phase1_template_hash": PHASE1_TEMPLATE_HASH,
    "vlm_hidden_dim":       CFG.vlm_hidden_dim,
    "n_samples_target":     _v1_total,
    "v1_cache_dir":         str(CFG.hidden_states_dir),
    "v1_manifest_path":     str(CFG.filtered_manifest_path),
}
_v2_meta_path = CFG.hidden_states_dir_v2 / "hidden_states_meta.json"
with open(_v2_meta_path, "w") as _f:
    _json.dump(_v2_meta, _f, indent=2)
print(f"v2 metadata written : {_v2_meta_path}")

print("\nok  v2 prep complete -- run S11.4b next.")


In [ ]:
# -- Move phase1_template_hash out of adapter_config.json so PEFT can load it -
# S10.2 embedded the hash directly into adapter_config.json. Newer PEFT
# (>= 0.10) raises TypeError when LoraConfig sees fields it doesn't know about.
# Fix: validate the hash, write it to a sidecar, then strip it from the
# adapter config. Idempotent -- safe to re-run.

import json as _json

_acfg_path    = CFG.phase1_adapter_dir / "adapter_config.json"
_sidecar_path = CFG.phase1_adapter_dir / "phase1_template_hash.json"

with open(_acfg_path, "r") as _f:
    _acfg = _json.load(_f)

_hash = _acfg.pop("phase1_template_hash", None)

if _hash is not None:
    # Drift check first -- if the hash on disk doesn't match the notebook's
    # current template, refuse to proceed; the v2 cache would be inconsistent.
    assert _hash == PHASE1_TEMPLATE_HASH, (
        f"Template hash drift:\n"
        f"  adapter_config.json  : {_hash!r}\n"
        f"  PHASE1_TEMPLATE_HASH : {PHASE1_TEMPLATE_HASH!r}\n"
        "Hidden states extracted now would not match the model's training-time prompt."
    )
    # Persist hash to a sidecar so future runs can still verify drift
    with open(_sidecar_path, "w") as _f:
        _json.dump({"phase1_template_hash": _hash}, _f, indent=2)
    # Rewrite adapter_config.json with the hash removed
    with open(_acfg_path, "w") as _f:
        _json.dump(_acfg, _f, indent=2)
    print(f"ok  stripped phase1_template_hash from adapter_config.json")
    print(f"    sidecar: {_sidecar_path}")
    print(f"    hash   : {_hash!r}")
else:
    print(f"adapter_config.json already clean -- no phase1_template_hash field.")
    if _sidecar_path.exists():
        with open(_sidecar_path, "r") as _f:
            print(f"sidecar present: {_json.load(_f)}")
    else:
        print(f"WARN: no sidecar at {_sidecar_path} -- drift check disabled "
              "until you re-create one.")

In [ ]:
# -- Load base Qwen2.5-VL-3B + saved LoRA adapter for extraction -------------
# This is the inference/extraction-time load (no fresh LoRA, no optimizer).
# It mirrors the quantization config used during training (S9.2) so the
# numerics of the forward pass match what produced the v1 cache.

import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

assert torch.cuda.is_available(), "CUDA not available -- extraction needs a GPU."

# 4-bit NF4 + BF16 compute -- identical to S9.2 training config
_bnb = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

print(f"Loading base model    : {CFG.vlm_model_id}")
_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG.vlm_model_id,
    quantization_config = _bnb,
    torch_dtype         = torch.bfloat16,
    device_map          = "auto",
)
print(f"  base hidden_size    : {_base.config.hidden_size}")
assert _base.config.hidden_size == CFG.vlm_hidden_dim, (
    f"base hidden_size {_base.config.hidden_size} != CFG.vlm_hidden_dim {CFG.vlm_hidden_dim}"
)

print(f"Loading LoRA adapter  : {CFG.phase1_adapter_dir}")
model = PeftModel.from_pretrained(
    _base,
    str(CFG.phase1_adapter_dir),
    is_trainable = False,   # extraction only -- no grads, no optimizer
)
model.eval()

# Confirm the adapter actually attached and at least one LoRA param is non-zero
_lora_params = [
    (n, p) for n, p in model.named_parameters() if "lora" in n.lower()
]
assert _lora_params, "No LoRA parameters found on model -- adapter did not attach."
_nonzero = sum(p.abs().sum().item() for _, p in _lora_params[:8])
assert _nonzero > 0, "LoRA params are all zero -- adapter likely loaded as randomly-initialised."
print(f"  LoRA tensors        : {len(_lora_params)} (sanity-check sum |W|={_nonzero:.3f})")

# Processor was saved alongside the adapter in S10.2
print(f"Loading processor     : {CFG.phase1_adapter_dir}")
processor = AutoProcessor.from_pretrained(str(CFG.phase1_adapter_dir))

# Final shape sanity: a tiny dummy forward pass to confirm last-hidden-state shape
with torch.no_grad():
    _dummy = processor.tokenizer("hello", return_tensors="pt").to(model.device)
    _out   = model(**_dummy, output_hidden_states=True)
    _hs    = _out.hidden_states[-1]
    assert _hs.shape[-1] == CFG.vlm_hidden_dim, (
        f"last hidden state dim {_hs.shape[-1]} != CFG.vlm_hidden_dim {CFG.vlm_hidden_dim}"
    )
print(f"  dummy forward       : last_hidden shape={tuple(_hs.shape)}  ok")
print("\nok  model + processor + adapter loaded.")

In [ ]:
# -- S11.4b  Run hidden-state extraction against the v2 cache ----------------
# Identical extraction logic as S11.3 but writes to hidden_states_dir_v2 and
# updates filtered_manifest_v2_path.  The v1 cache + v1 manifest are NOT
# touched.  Restart-safe: re-running resumes from the next uncached sample.
#
# Pre-conditions (verified by S11.4a, but re-checked here for safety):
#   - v2 manifest exists and has the right length.
#   - The model in memory is the *resumed* adapter (caller's responsibility --
#     we cannot tell from in-memory state whether weights have been updated).
#
# Expected runtime: same as S11.3 -- ~15-30 s/sample on T4, faster on A100.
# Pickier checks at the end ensure shard <-> manifest consistency before
# Phase 2 begins consuming the cache.

import random as _random
import torch as _torch

print("=" * 60)
print("S11.4b  Running VLM hidden-state extraction (v2 cache)")
print("=" * 60)

# -- 1. Re-validate v2 manifest ---------------------------------------------
assert CFG.filtered_manifest_v2_path.exists(), (
    f"v2 manifest not found at {CFG.filtered_manifest_v2_path}. "
    "Run S11.4a first."
)
_manifest_v2 = load_json(CFG.filtered_manifest_v2_path)
print(f"v2 manifest         : {len(_manifest_v2)} samples")

# -- 2. Rebuild raw samples if not in scope (deterministic from manifest) ----
# build_vlm_dataset reads samples_filtered.json (v1) -- the underlying sample
# data is identical between v1 and v2; only shard_id metadata differs.
if "_full_raw" not in dir() or _full_raw is None:
    print("Rebuilding _full_raw from v1 manifest (sample data is shared)...")
    _full_raw = build_vlm_dataset(
        manifest_path    = CFG.filtered_manifest_path,
        data_dir         = CFG.data_dir,
        sample_limit     = None,
        warn_on_fallback = False,
    )
print(f"raw samples loaded  : {len(_full_raw)}")
assert len(_full_raw) == len(_manifest_v2), (
    f"_full_raw length {len(_full_raw)} != v2 manifest length {len(_manifest_v2)}. "
    "build_vlm_dataset must produce one entry per manifest row (in order)."
)

# Safety: confirm sample_ids align in order between _full_raw and _manifest_v2
_misaligned = [
    (i, _full_raw[i]["sample_id"], _manifest_v2[i].get("sample_id"))
    for i in range(len(_full_raw))
    if _full_raw[i]["sample_id"] != _manifest_v2[i].get("sample_id")
]
assert not _misaligned, (
    f"sample_id alignment failed at {len(_misaligned)} positions. "
    f"first mismatch: {_misaligned[0]}"
)

# -- 3. Run extraction -------------------------------------------------------
_stats_v2 = cache_vlm_hidden_states(
    raw_samples       = _full_raw,
    manifest          = _manifest_v2,
    model             = model,
    processor         = processor,
    hidden_states_dir = CFG.hidden_states_dir_v2,
    manifest_path     = CFG.filtered_manifest_v2_path,
    shard_size        = CFG.phase2_shard_size,    # explicit; do not rely on hs_shard_size attr
    device            = "cuda",
)

print("\n--- Extraction summary (v2) ---")
for k, v in _stats_v2.items():
    print(f"  {k:<20s}: {v}")

# -- 4. Verify shards on disk ------------------------------------------------
_shard_files_v2 = sorted(CFG.hidden_states_dir_v2.glob("shard_*.pt"))
assert _shard_files_v2, f"No shard files in {CFG.hidden_states_dir_v2}."
print(f"\nVerifying {len(_shard_files_v2)} shard files...")
_total_rows_v2 = 0
for _sf in _shard_files_v2:
    _data = _torch.load(_sf, map_location="cpu", weights_only=False)
    assert "hidden_states" in _data and "sample_ids" in _data, (
        f"Shard {_sf.name} missing keys: {list(_data.keys())}"
    )
    _hs = _data["hidden_states"]
    assert _hs.ndim == 2 and _hs.shape[1] == CFG.vlm_hidden_dim, (
        f"Shard {_sf.name}: shape {tuple(_hs.shape)} != (N, {CFG.vlm_hidden_dim})"
    )
    assert _hs.dtype == _torch.float32, f"Shard {_sf.name}: dtype {_hs.dtype}"
    assert len(_data["sample_ids"]) == _hs.shape[0], (
        f"Shard {_sf.name}: sample_ids/rows length mismatch"
    )
    assert not _hs.isnan().any(), f"Shard {_sf.name} contains NaN"
    _total_rows_v2 += _hs.shape[0]
    print(f"  ok  {_sf.name}: shape={tuple(_hs.shape)}, dtype={_hs.dtype}")
print(f"  Total rows across v2 shards: {_total_rows_v2}")

# -- 5. Verify manifest coverage --------------------------------------------
_manifest_v2 = load_json(CFG.filtered_manifest_v2_path)   # reload from disk
_n_shard     = sum(1 for m in _manifest_v2 if m.get("shard_id") is not None)
_n_no_shard  = sum(1 for m in _manifest_v2 if m.get("shard_id") is None)
print(f"\nManifest coverage (v2):")
print(f"  with shard_id     : {_n_shard}")
print(f"  without shard_id  : {_n_no_shard}  (extraction failures -- skipped in Phase 2)")
if _n_no_shard > 0:
    log.warning(
        f"{_n_no_shard} v2 manifest entries still missing shard_id. "
        "These are samples where extract_hidden_state_for_sample returned None. "
        "Verify counts match _stats_v2['skipped'] before proceeding to Phase 2."
    )
    assert _n_no_shard == _stats_v2["skipped"], (
        f"Bookkeeping mismatch: {_n_no_shard} uncached entries vs "
        f"{_stats_v2['skipped']} reported skips."
    )

# -- 6. Round-trip test on a random cached entry -----------------------------
_cached_entries = [m for m in _manifest_v2 if m.get("shard_id") is not None]
assert _cached_entries, "no cached entries to round-trip"
_pick      = _random.choice(_cached_entries)
_shard_path = CFG.hidden_states_dir_v2 / f"{_pick['shard_id']}.pt"
assert _shard_path.exists(), f"shard {_shard_path} referenced but missing on disk"
_data = _torch.load(_shard_path, map_location="cpu", weights_only=False)
_row  = _pick["row_index"]
assert _data["sample_ids"][_row] == _pick["sample_id"], (
    f"round-trip mismatch: manifest={_pick['sample_id']!r}, "
    f"shard row {_row}={_data['sample_ids'][_row]!r}"
)
_rt_hs = _data["hidden_states"][_row]
assert _rt_hs.shape == (CFG.vlm_hidden_dim,), (
    f"round-trip hidden state shape {_rt_hs.shape} != ({CFG.vlm_hidden_dim},)"
)
print(f"\nRound-trip ok: sample_id={_pick['sample_id']!r}")
print(f"  shard={_pick['shard_id']}, row={_row}")
print(f"  hidden_state shape={tuple(_rt_hs.shape)}, dtype={_rt_hs.dtype}")

# -- 7. Final summary --------------------------------------------------------
print("\n" + "=" * 60)
print("v2 cache complete.")
print(f"  shards dir   : {CFG.hidden_states_dir_v2}")
print(f"  manifest     : {CFG.filtered_manifest_v2_path}")
print(f"  metadata     : {CFG.hidden_states_dir_v2 / 'hidden_states_meta.json'}")
print("Phase 2: point your dataloader at filtered_manifest_v2_path "
      "and load shards from hidden_states_dir_v2.")
print("=" * 60)


In [ ]:
# -- S11.4b v2 Run hidden-state extraction against the v2 cache (corrected) ----
# Same logic as S11.3 but writes to hidden_states_dir_v2 and updates the
# v2 manifest only.  v1 cache and v1 manifest are NOT touched.
# Restart-safe: re-running resumes from the next uncached sample.
#
# v1.6 fix: build_vlm_dataset legitimately filters out samples (e.g. ones
# select_target_bbox can't resolve), so len(_full_raw) <= len(manifest_v2)
# is normal.  cache_vlm_hidden_states does sample_id-based dispatch, so we
# only need every _full_raw sample_id to exist in the v2 manifest -- not
# positional alignment.

import random as _random
import torch as _torch

print("=" * 60)
print("S11.4b  Running VLM hidden-state extraction (v2 cache)")
print("=" * 60)

# -- 1. Re-validate v2 manifest ----------------------------------------------
assert CFG.filtered_manifest_v2_path.exists(), (
    f"v2 manifest not found at {CFG.filtered_manifest_v2_path}. Run S11.4a first."
)
_manifest_v2 = load_json(CFG.filtered_manifest_v2_path)
_n_manifest  = len(_manifest_v2)
print(f"v2 manifest         : {_n_manifest} samples")

# -- 2. Rebuild raw samples if not in scope ----------------------------------
if "_full_raw" not in dir() or _full_raw is None:
    print("Rebuilding _full_raw from v1 manifest...")
    _full_raw = build_vlm_dataset(
        manifest_path    = CFG.filtered_manifest_path,
        data_dir         = CFG.data_dir,
        sample_limit     = None,
        warn_on_fallback = False,
    )
_n_raw = len(_full_raw)
print(f"raw samples loaded  : {_n_raw}")
print(f"  (manifest entries not in _full_raw: {_n_manifest - _n_raw} -- "
      "these will be left uncached, same as v1)")

# Every _full_raw sample_id must be in the v2 manifest -- sample_id-based
# dispatch in cache_vlm_hidden_states will RuntimeError otherwise. Subset
# relationship is OK (we don't require equality).
_manifest_ids = {m["sample_id"] for m in _manifest_v2}
_orphans = [rs["sample_id"] for rs in _full_raw if rs["sample_id"] not in _manifest_ids]
assert not _orphans, (
    f"{len(_orphans)} sample_ids in _full_raw are not in the v2 manifest. "
    f"first orphan: {_orphans[0]!r}"
)

# -- 3. Run extraction -------------------------------------------------------
_stats_v2 = cache_vlm_hidden_states(
    raw_samples       = _full_raw,
    manifest          = _manifest_v2,
    model             = model,
    processor         = processor,
    hidden_states_dir = CFG.hidden_states_dir_v2,
    manifest_path     = CFG.filtered_manifest_v2_path,
    shard_size        = CFG.phase2_shard_size,
    device            = "cuda",
)

print("\n--- Extraction summary (v2) ---")
for k, v in _stats_v2.items():
    print(f"  {k:<20s}: {v}")

# -- 4. Verify shards on disk ------------------------------------------------
_shard_files_v2 = sorted(CFG.hidden_states_dir_v2.glob("shard_*.pt"))
assert _shard_files_v2, f"No shard files in {CFG.hidden_states_dir_v2}."
print(f"\nVerifying {len(_shard_files_v2)} shard files...")
_total_rows_v2 = 0
for _sf in _shard_files_v2:
    _data = _torch.load(_sf, map_location="cpu", weights_only=False)
    assert "hidden_states" in _data and "sample_ids" in _data, (
        f"Shard {_sf.name} missing keys: {list(_data.keys())}"
    )
    _hs = _data["hidden_states"]
    assert _hs.ndim == 2 and _hs.shape[1] == CFG.vlm_hidden_dim, (
        f"Shard {_sf.name}: shape {tuple(_hs.shape)} != (N, {CFG.vlm_hidden_dim})"
    )
    assert _hs.dtype == _torch.float32, f"Shard {_sf.name}: dtype {_hs.dtype}"
    assert len(_data["sample_ids"]) == _hs.shape[0], (
        f"Shard {_sf.name}: sample_ids/rows length mismatch"
    )
    assert not _hs.isnan().any(), f"Shard {_sf.name} contains NaN"
    _total_rows_v2 += _hs.shape[0]
    print(f"  ok  {_sf.name}: shape={tuple(_hs.shape)}, dtype={_hs.dtype}")
print(f"  Total rows across v2 shards: {_total_rows_v2}")

# -- 5. Verify manifest coverage (corrected accounting) ----------------------
_manifest_v2 = load_json(CFG.filtered_manifest_v2_path)
_n_shard    = sum(1 for m in _manifest_v2 if m.get("shard_id") is not None)
_n_no_shard = sum(1 for m in _manifest_v2 if m.get("shard_id") is None)
print(f"\nManifest coverage (v2):")
print(f"  with shard_id     : {_n_shard}")
print(f"  without shard_id  : {_n_no_shard}")
print(f"    breakdown:")
print(f"      not in _full_raw  : {_n_manifest - _n_raw}")
print(f"      extraction failed : {_stats_v2['skipped']}")

# Expected: uncached = (manifest_size - full_raw_size) + extraction_failures
# (assumes a fresh v2 run; if resuming a partial v2, cached_before > 0 too)
_expected_no_shard = (_n_manifest - _n_raw) + _stats_v2["skipped"]
# Account for resume case where some entries were already cached coming in:
# extracted + cached_before should equal _n_shard.
_expected_shard = _stats_v2["extracted"] + _stats_v2["cached_before"]
assert _n_shard == _expected_shard, (
    f"shard_id-set mismatch: manifest has {_n_shard} cached but stats say "
    f"extracted={_stats_v2['extracted']} + cached_before={_stats_v2['cached_before']} "
    f"= {_expected_shard}"
)
assert _n_no_shard == _expected_no_shard, (
    f"shard_id-None mismatch: manifest has {_n_no_shard} uncached but expected "
    f"(manifest_size - full_raw_size) + skipped = "
    f"({_n_manifest} - {_n_raw}) + {_stats_v2['skipped']} = {_expected_no_shard}"
)
print(f"  -> coverage accounting checks out.")

# -- 6. Round-trip test ------------------------------------------------------
_cached_entries = [m for m in _manifest_v2 if m.get("shard_id") is not None]
assert _cached_entries, "no cached entries to round-trip"
_pick      = _random.choice(_cached_entries)
_shard_path = CFG.hidden_states_dir_v2 / f"{_pick['shard_id']}.pt"
assert _shard_path.exists(), f"shard {_shard_path} referenced but missing on disk"
_data = _torch.load(_shard_path, map_location="cpu", weights_only=False)
_row  = _pick["row_index"]
assert _data["sample_ids"][_row] == _pick["sample_id"], (
    f"round-trip mismatch: manifest={_pick['sample_id']!r}, "
    f"shard row {_row}={_data['sample_ids'][_row]!r}"
)
_rt_hs = _data["hidden_states"][_row]
assert _rt_hs.shape == (CFG.vlm_hidden_dim,)
print(f"\nRound-trip ok: sample_id={_pick['sample_id']!r}")
print(f"  shard={_pick['shard_id']}, row={_row}")
print(f"  hidden_state shape={tuple(_rt_hs.shape)}, dtype={_rt_hs.dtype}")

# -- 7. Final summary --------------------------------------------------------
print("\n" + "=" * 60)
print("v2 cache complete.")
print(f"  shards dir   : {CFG.hidden_states_dir_v2}")
print(f"  manifest     : {CFG.filtered_manifest_v2_path}")
print(f"  metadata     : {CFG.hidden_states_dir_v2 / 'hidden_states_meta.json'}")
print("Phase 2: point your dataloader at filtered_manifest_v2_path "
      "and load shards from hidden_states_dir_v2.")
print("=" * 60)

# Section 12 -- Phase 1 End-to-End Smoke Test

This section is the final mandatory gate before Phase 2.

It verifies all five Phase 1 components are functioning correctly in sequence:

1. **Dataset pipeline** -- `build_vlm_dataset()` produces valid sample dicts with
   `source_image_path`, `instruction`, `target_json`, `bbox`, `edit_type`
2. **Tokenization** -- `tokenize_sample()` produces correct tensor shapes and non-empty labels
3. **DataCollator** -- `Phase1DataCollator` produces correct batched shapes with
   pixel_values concatenated (not stacked) and image_grid_thw matching batch size
4. **VLM forward pass** -- model produces non-NaN loss with correct logits shape
5. **Hidden-state extraction** -- `extract_hidden_state_for_sample()` produces
   `(hidden_dim,)` float32 tensor with correct shape and no NaN values

This smoke test uses a fresh set of samples and does NOT rely on prior cell state,
so it can be run independently to verify the full pipeline is intact after any
notebook restart or code change.


In [ ]:
# -- S12.1  Full Phase 1 end-to-end smoke test --------------------------------
# Tests all 5 Phase 1 components using 4 samples from the filtered manifest.
# Re-runnable independently of prior cells -- rebuilds all state from scratch.
# Deliberately uses a DIFFERENT set of samples than the shape-validation test
# in S9.6 to catch any per-sample edge cases.
#
# Pass/fail: any assertion failure raises immediately with diagnostic context.
# All 5 components must pass before Phase 2 implementation begins.

import json as _json_mod

print("=" * 62)
print("S12.1  Phase 1 End-to-End Smoke Test")
print("=" * 62)
print()

_SMOKE_N = 4   # samples to test; 4 gives batch size >= 2 after sentinel filtering

# ============================================================
# Component 1: Dataset pipeline
# ============================================================
print("--- Component 1: Dataset pipeline ---")

_smoke_raw = build_vlm_dataset(
    manifest_path  = CFG.filtered_manifest_path,
    data_dir       = CFG.data_dir,
    sample_limit   = _SMOKE_N * 2,   # extra headroom in case some samples are skipped
    warn_on_fallback = True,
)

assert len(_smoke_raw) >= _SMOKE_N, (
    f"build_vlm_dataset returned only {len(_smoke_raw)} samples "
    f"(needed >= {_SMOKE_N}). Check filtered_manifest_path."
)

_required_keys = {
    "sample_id", "edit_type", "instruction", "bbox", "bbox_matched_by",
    "target_json", "source_image_path", "annotations", "seg_dims",
}
for _i, _s in enumerate(_smoke_raw[:_SMOKE_N]):
    _missing = _required_keys - _s.keys()
    assert not _missing, (
        f"Sample {_i} missing keys: {_missing}\n"
        f"Sample id: {_s.get('sample_id','?')}"
    )
    # Validate bbox is 4 numbers in [0, 1000]
    _bbox = _s["bbox"]
    assert isinstance(_bbox, list) and len(_bbox) == 4, (
        f"Sample {_i}: bbox must be list of 4, got {_bbox!r}"
    )
    assert all(0 <= v <= 1000 for v in _bbox), (
        f"Sample {_i}: bbox coords out of [0,1000]: {_bbox}"
    )
    # Validate target_json is parseable
    try:
        _parsed_json = _json_mod.loads(_s["target_json"])
    except Exception as _e:
        raise AssertionError(
            f"Sample {_i}: target_json is not valid JSON: {_e}\n"
            f"  target_json: {_s['target_json']!r}"
        )
    # Validate source image exists
    assert Path(_s["source_image_path"]).exists(), (
        f"Sample {_i}: source_image_path does not exist: {_s['source_image_path']}"
    )

print(f"  ok  {_SMOKE_N} samples validated (keys, bbox, target_json, image paths)")

# ============================================================
# Component 2: Tokenization
# ============================================================
print("\n--- Component 2: Tokenization (tokenize_sample) ---")

_smoke_tokenized = []
for _i, _rs in enumerate(_smoke_raw[:_SMOKE_N]):
    _tok = tokenize_sample(_rs, processor, max_length=CFG.phase1_max_seq_len)
    if _tok is None:
        log.warning(f"Smoke test sample {_i} ({_rs['sample_id']}) skipped by tokenize_sample")
        continue

    # Shape checks
    _seq_len = _tok["input_ids"].shape[0]
    assert _tok["attention_mask"].shape == (_seq_len,), (
        f"Sample {_i}: attention_mask shape {_tok['attention_mask'].shape} "
        f"!= input_ids shape ({_seq_len},)"
    )
    assert _tok["labels"].shape == (_seq_len,), (
        f"Sample {_i}: labels shape {_tok['labels'].shape} != ({_seq_len},)"
    )
    _n_resp = (_tok["labels"] != -100).sum().item()
    assert _n_resp > 0, (
        f"Sample {_i}: all labels are -100 (no response tokens). "
        "Check response_start detection in tokenize_sample."
    )
    # pixel_values should exist for image inputs
    assert _tok.get("pixel_values") is not None, (
        f"Sample {_i}: pixel_values is None -- image may not have loaded"
    )
    _smoke_tokenized.append(_tok)
    print(f"  ok  sample {_i}: seq_len={_seq_len}, resp_tokens={_n_resp}, "
          f"pixel_values={tuple(_tok['pixel_values'].shape)}")

assert len(_smoke_tokenized) >= 2, (
    f"Only {len(_smoke_tokenized)} samples tokenized successfully (need >= 2). "
    "Cannot build a valid batch. Check tokenize_sample logs."
)

# ============================================================
# Component 3: DataCollator
# ============================================================
print(f"\n--- Component 3: DataCollator (batch of {len(_smoke_tokenized)}) ---")

_collator   = Phase1DataCollator(processor)
_smoke_batch = _collator(_smoke_tokenized)
_B           = len(_smoke_tokenized)

assert _smoke_batch["input_ids"].shape[0]    == _B, "input_ids batch dim"
assert _smoke_batch["attention_mask"].shape  == _smoke_batch["input_ids"].shape, \
    "attention_mask shape mismatch"
assert _smoke_batch["labels"].shape          == _smoke_batch["input_ids"].shape, \
    "labels shape mismatch"

_pv = _smoke_batch.get("pixel_values")
assert _pv is not None, "pixel_values missing from collated batch"
assert _pv.ndim >= 2, f"pixel_values ndim={_pv.ndim}, expected >= 2"

_gthw = _smoke_batch.get("image_grid_thw")
assert _gthw is not None, "image_grid_thw missing from collated batch"
assert _gthw.shape[0] == _B, (
    f"image_grid_thw.shape[0]={_gthw.shape[0]} != B={_B}. "
    "Each sample must have exactly 1 source image."
)

print(f"  ok  input_ids      : {tuple(_smoke_batch['input_ids'].shape)}")
print(f"  ok  attention_mask : {tuple(_smoke_batch['attention_mask'].shape)}")
print(f"  ok  labels         : {tuple(_smoke_batch['labels'].shape)}")
print(f"  ok  pixel_values   : {tuple(_pv.shape)}")
print(f"  ok  image_grid_thw : {tuple(_gthw.shape)}")

# ============================================================
# Component 4: VLM forward pass
# ============================================================
print(f"\n--- Component 4: VLM forward pass ---")

model.eval()
_dev = str(next(model.parameters()).device)
_b_dev = {k: v.to(_dev) if isinstance(v, torch.Tensor) else v
          for k, v in _smoke_batch.items()}

with torch.no_grad():
    _fwd_out = model(**_b_dev)

assert _fwd_out.loss is not None, "Model returned no loss -- check labels tensor"
assert not _fwd_out.loss.isnan().item(), (
    f"NaN loss in forward pass. "
    f"Causes: NaN pixel_values, all-masked labels, or BF16 overflow."
)
_expected_logits_shape = (_B, _smoke_batch["input_ids"].shape[1], model.config.vocab_size)
assert _fwd_out.logits.shape == _expected_logits_shape, (
    f"Unexpected logits shape: {_fwd_out.logits.shape} "
    f"!= expected {_expected_logits_shape}"
)

print(f"  ok  loss   : {_fwd_out.loss.item():.4f}")
print(f"  ok  logits : {tuple(_fwd_out.logits.shape)}")
model.train()

# ============================================================
# Component 5: Hidden-state extraction
# ============================================================
print(f"\n--- Component 5: Hidden-state extraction ---")

model.eval()
_hs_results = []
for _i, _rs in enumerate(_smoke_raw[:_SMOKE_N]):
    _hs = extract_hidden_state_for_sample(_rs, model, processor, "cuda")
    if _hs is None:
        log.warning(f"  Smoke test sample {_i} skipped in hidden-state extraction")
        continue

    assert _hs.shape == (CFG.vlm_hidden_dim,), (
        f"Sample {_i}: hidden state shape {_hs.shape} != ({CFG.vlm_hidden_dim},)"
    )
    assert _hs.dtype == torch.float32, (
        f"Sample {_i}: expected float32, got {_hs.dtype}"
    )
    assert not _hs.isnan().any(), (
        f"Sample {_i}: NaN values in hidden state"
    )
    _hs_results.append(_hs)
    print(f"  ok  sample {_i}: shape={tuple(_hs.shape)}, "
          f"range=[{_hs.min():.3f}, {_hs.max():.3f}]")

assert len(_hs_results) >= 2, (
    f"Only {len(_hs_results)} hidden states extracted successfully (need >= 2). "
    "Check extract_hidden_state_for_sample logs."
)
model.train()

# ============================================================
# Final summary
# ============================================================
print()
print("=" * 62)
print("ok  S12.1 ALL 5 COMPONENTS PASSED")
print()
print(f"  1. Dataset pipeline    : {_SMOKE_N} samples validated")
print(f"  2. Tokenization        : {len(_smoke_tokenized)}/{_SMOKE_N} tokenized successfully")
print(f"  3. DataCollator        : batch shape {tuple(_smoke_batch['input_ids'].shape)} ok")
print(f"  4. VLM forward pass    : loss={_fwd_out.loss.item():.4f}, logits ok")
print(f"  5. Hidden-state extr.  : {len(_hs_results)}/{_SMOKE_N} extracted, shape ({CFG.vlm_hidden_dim},)")
print()
print(f"  PHASE1_TEMPLATE_HASH   : {PHASE1_TEMPLATE_HASH}")
print(f"  Adapter saved at       : {CFG.phase1_adapter_dir}")
print(f"  Hidden states at       : {CFG.hidden_states_dir}")
print()
print("Phase 1 complete. Proceed to Phase 2 (SD inpainting bridge training).")
print("=" * 62)
